# XAI-Based Welfare Policy Recommendation Chatbot

실행용 노트북이다. 같은 문항의 시간·메모리 측정은 `Benchmark_XAI_Based_Welfare_Policy_Recommendation_Chatbot.ipynb`를 사용한다.

대상 GPU는 Google Colab G4, `NVIDIA RTX PRO 6000 Blackwell Server Edition`, 96GB이다. 답변 생성과 Captum을 한 GPU에 두면서 OOM을 피하기 위해 이 용량을 쓴다.

## 프로젝트 개요

Qdrant 벡터 검색과 `Dongjin-kr/ko-reranker` 재정렬로 관련 복지 정책 문서를 찾고, `Qwen3-4B-Instruct-2507`이 자연어 상담 답변을 생성한 뒤, Captum Feature Ablation으로 문서와 질문 단어의 기여도를 함께 제시한다.

## RAG 구조

1. 사용자 질문 입력
2. Qdrant 통합 컬렉션에서 dense vector search로 후보 문서 검색
3. `Dongjin-kr/ko-reranker`로 질문·문서 관련성 재평가 (512토큰 1회)
4. reranker 점수 순 정렬, 정책별 최대 900자 문맥 구성
5. vLLM이 자연어 정책 상담 답변 생성 (포트 8001)
6. Captum 문서·질문 단어 Feature Ablation 결과를 웹 UI에 전달

## 설계 의도

1차 검색에서 후보를 확보하고 2차 reranker로 관련성을 평가하는 2-stage RAG를 사용한다.
지역 가산점 없이 reranker 점수 순으로 정렬하며, 정책명과 URL 기준으로 중복을 묶는다.
정책별 문맥은 900자로 제한하고, Captum 분석은 생성된 답변 전체를 대상으로 한다.
추론 중 자동 LoRA 학습은 하지 않으며, 저장된 어댑터가 있으면 병합해 vLLM에 넘긴다.
답변 생성은 vLLM이 담당하고, 기여도는 HF SDPA에서 Feature Ablation으로 계산한다.

### 환경

- Google Colab G4 GPU (`NVIDIA RTX PRO 6000 Blackwell Server Edition`, 96GB)
- Python·PyTorch: Colab 기본 환경을 유지한다. 답변 생성에 FlashAttention-2를 쓰지 않으므로 FA2 wheel은 기본 설치하지 않는다
- LLM: `Qwen/Qwen3-4B-Instruct-2507`
- Embedding: `BAAI/bge-m3`
- Reranker: `Dongjin-kr/ko-reranker`
- Vector DB: Qdrant, 통합 컬렉션 `welfare_policies_bge_m3`
- LoRA: 저장된 어댑터가 있으면 병합해 vLLM이 로드한다
- RAG/XAI API: FastAPI, Colab 포트 9000 (`127.0.0.1`)
- 답변 생성 서빙: vLLM 0.19.* (격리 `virtualenv`, `vllm serve`, 포트 8001)
- XAI: Hugging Face 모델 + PyTorch SDPA
- Frontend: GitHub `app/` 웹 UI, Colab 포트 8000
- Tunnel: Cloudflare Tunnel → `http://localhost:8000`
- 공개 UI: https://kr-welfare-xai.com/
- 필수 설정: `QDRANT_URL`, `QDRANT_API_KEY`, `TUNNEL_TOKEN`

## XAI 분석 방식 (Captum)

- **문서 단위 기여도 — FeatureAblation**: 선택된 정책 문서를 하나씩 마스킹하고 답변 로그확률이 얼마나 떨어지는지 측정한다. `perturbations_per_eval=1`로 한 번에 하나만 계산한다
- **질문 단어 단위 기여도 — FeatureAblation**: 질문 단어를 하나씩 마스킹하고 같은 로그확률 변화를 측정한다
- **logits 범위**: 답변 토큰 구간에만 `lm_head`를 적용한다. 손실은 float32로 계산한다
- **메모리**: 분석이 끝나면 CUDA 캐시를 반환해 vLLM과 GPU를 나눠 쓴다
- **형태소 분석 — Kiwi(kiwipiepy)**: 질문을 의미 단위로 분리해 원문 하이라이트 위치를 연결한다
- **`rag_cache`**: `/analyze`가 `/ask` 직후면 검색·재정렬·생성 결과를 재사용한다
- reranker logit은 질문-문서 쌍마다 sigmoid로 0~1에 둔다. 정렬과 정책 묶음은 이 값을 쓰고, raw logit은 `reranker_score_raw`에 남긴다
- 기여도 백분율은 양수 Feature Ablation 값의 합으로 만든다. 음수는 0으로 두고, 양수가 없으면 기여도도 0이다

## 생성 경로 (vLLM)

- `/ask`만 vLLM을 사용한다. `/analyze`는 HF SDPA에서 Feature Ablation만 실행한다
- 웹 UI는 GitHub `app/`을 clone한다. `/chat`이 Colab `/ask`만 기다린 뒤 답을 먼저 보여주고, `/captum`이 `/analyze`로 근거를 붙인다
- vLLM은 메인 커널과 분리된 `virtualenv`에 설치하고 `vllm serve`로만 기동한다. ping 스모크는 넣지 않는다. 준비 여부는 `/v1/models`로 확인한다
- 새 세션에서는 Drive의 pip 캐시와 `cache/vllm/venv_*_vllm019.tar.gz`를 `/content/vllm_env`로 복원한다. venv를 Drive에서 직접 실행하지는 않는다
- 답변용 HF 모델은 로드하지 않는다 (`SKIP_ANSWER_MODEL_LOAD=True`)
- 같은 GPU를 XAI와 vLLM이 나눠 쓰므로 `--gpu-memory-utilization` 기본값은 0.30이다. 96GB에서도 우선 이 값으로 두고, 여유를 확인한 뒤에만 올린다

생성 설정: `do_sample=True`, `temperature=0.3`, `top_p=0.9`, `repetition_penalty=1.1`, `max_tokens=700`.

## 참고 자료 (코드 구현용)

- BAAI/bge-m3: https://huggingface.co/BAAI/bge-m3
- Captum Feature Ablation: https://captum.ai/api/feature_ablation.html
- Cloudflare Tunnel: https://developers.cloudflare.com/tunnel/
- Dongjin-kr/ko-reranker: https://huggingface.co/Dongjin-kr/ko-reranker
- FastAPI: https://fastapi.tiangolo.com/tutorial/
- Kiwi / kiwipiepy: https://github.com/bab2min/kiwipiepy
- PEFT / LoRA: https://huggingface.co/docs/peft/main/index
- PyTorch SDPA: https://docs.pytorch.org/tutorials/intermediate/transformer_building_blocks.html
- Qdrant / cosine similarity: https://qdrant.tech/documentation/search/search/
- Qwen3-4B-Instruct-2507: https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507
- vLLM: https://docs.vllm.ai/en/latest/serving/online_serving/

## 노트북 셀 번호 (위→아래, 총 26개)

| 셀 | 내용 | 실행 필수 |
|---:|------|:--:|
| 1 | 설명 (이 셀) | |
| 2 | Colab GPU·Python·PyTorch 환경 확인 | 예 |
| 3 | Drive 연결·경로·HF/pip 캐시 설정 | 예 |
| 4 | 패키지 설치·cloudflared 준비 (FA2는 기본 생략) | 예 |
| 5 | 설정값 로드 | 예 |
| 6 | 설정·로컬 경로 검증 | 예 |
| 7 | LLM 로컬 경로 확인 | 선택 |
| 8 | XAI SDPA 모델과 저장된 LoRA 로드. 답변 HF 모델은 로드하지 않음 | 예 |
| 9 | LoRA 사용 상태 안내. 자동 재학습 없음 | 예 |
| 10 | Embedding 모델 로드 | 예 |
| 11 | Qdrant 연결·벡터 구성 검증 | 예 |
| 12 | Korean Reranker 로드 | 예 |
| 13 | 통합 payload 정제 | 예 |
| 14 | Qdrant 1차 후보 검색 | 예 |
| 15 | 512토큰 rerank·지역 가산 없음 | 예 |
| 16 | 정책 묶음·900자 문맥·최종 context 구성 | 예 |
| 17 | 프롬프트·RAG 답변 함수 (생성은 셀 20에서 vLLM로 연결) | 예 |
| **18** | **vLLM — LoRA 병합 체크포인트 저장** | 예 |
| **19** | **vLLM — 가상환경 준비와 서버 기동** | 예 |
| **20** | **vLLM — `generate_with_qwen` 정의** | 예 |
| 21 | Frontend 경로·환경 확인 | 예 |
| 22 | Frontend sparse clone | 예 |
| 23 | Frontend 런타임 패치 | 예 |
| **24** | **RAG API 9000 및 Captum 함수** | 예 |
| **25** | **Frontend 8000** | 예 |
| **26** | **Cloudflare Tunnel** | 예 |

실행: 셀 2→17 → 18→20 → 21→23 → 24→26 → 브라우저.

셀 20 전에 `/ask`를 호출하지 않는다. 답변 생성은 vLLM, Captum은 SDPA이다.


In [1]:
# [셀 02/26] Colab GPU 런타임 확인 (torch는 노트북에서 교체하지 않음)
import platform
import sys
import torch

try:
    import google.colab  # noqa: F401
except ImportError as exc:
    raise RuntimeError("Google Colab 전용 노트북입니다. Colab GPU 런타임에서 실행하세요.") from exc

BOOT_ENV = {
    "python": platform.python_version(),
    "python_tag": f"cp{sys.version_info.major}{sys.version_info.minor}",
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "platform": platform.platform(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(BOOT_ENV)
if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임을 선택한 뒤 다시 실행해 주세요.")

gpu_name = torch.cuda.get_device_name(0)
gpu_bytes = int(torch.cuda.get_device_properties(0).total_memory)
ok_name = ("RTX PRO 6000" in gpu_name) or ("Blackwell" in gpu_name and "6000" in gpu_name)
ok_mem = 80 * (2 ** 30) <= gpu_bytes <= 110 * (2 ** 30)
if not ok_name or not ok_mem:
    raise RuntimeError(
        "이 노트북은 Colab G4(NVIDIA RTX PRO 6000 Blackwell Server Edition, 96GB)용입니다. "
        f"실제 GPU: {gpu_name!r}, bytes={gpu_bytes}"
    )
print("GPU 확인:", gpu_name, f"{gpu_bytes / (2 ** 30):.1f} GiB")


{'python': '3.13.15', 'python_tag': 'cp313', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39', 'gpu': 'NVIDIA RTX PRO 6000 Blackwell Server Edition'}
GPU 확인: NVIDIA RTX PRO 6000 Blackwell Server Edition 95.0 GiB


In [2]:
# ============================================================
# [셀 03/26] Drive 연결 및 고정 작업 경로 / HF cache 설정 / LLM으로 생성
# ============================================================

import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# 작업 폴더
PROJECT_PATH = Path("/content/drive/MyDrive/Shared_Project")
PROJECT_PATH.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_PATH)

# 캐시/로그/모델 경로
HF_HOME = PROJECT_PATH / "hf_cache"
# HF_HOME = Path("/content/hf_cache")  # colab 로컬 저장소
HF_HUB_CACHE = HF_HOME / "hub"
LOCAL_MODEL_DIR = PROJECT_PATH / "local_models"
LOG_DIR = PROJECT_PATH / "logs"

for directory in (HF_HOME, HF_HUB_CACHE, LOCAL_MODEL_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_HUB_CACHE)

PIP_CACHE_DIR = PROJECT_PATH / "cache" / "pip"
PIP_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE_DIR)
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"

print("작업 위치:", PROJECT_PATH)
print("HF cache:", HF_HOME)
print("Local model dir:", LOCAL_MODEL_DIR)
print("Log dir:", LOG_DIR)
print("pip cache:", PIP_CACHE_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
작업 위치: /content/drive/MyDrive/Shared_Project
HF cache: /content/drive/MyDrive/Shared_Project/hf_cache
Local model dir: /content/drive/MyDrive/Shared_Project/local_models
Log dir: /content/drive/MyDrive/Shared_Project/logs
pip cache: /content/drive/MyDrive/Shared_Project/cache/pip


In [3]:
# [셀 04/26] Colab 런타임 기준 패키지 pin. FA2 wheel은 기본 생략
import hashlib
import importlib
import importlib.metadata
import importlib.util
import json
import os
import re
import subprocess
import sys
import urllib.error
import urllib.parse
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

missing = [key for key in ("PROJECT_PATH", "BOOT_ENV", "torch") if key not in globals()]
if missing:
    raise RuntimeError(f"선행 셀 02·03을 먼저 실행해 주세요. 누락: {missing}")

INSTALL_LOG_DIR = PROJECT_PATH / "logs" / "xai_chatbot"
INSTALL_LOG_DIR.mkdir(parents=True, exist_ok=True)
INSTALL_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
REQUIRE_FLASH_ATTN = os.getenv("REQUIRE_FLASH_ATTN", "0").lower() in {"1", "true", "yes"}
FA2_ENABLE_AUTO_SEARCH = os.getenv("FA2_ENABLE_AUTO_SEARCH", "1").lower() in {"1", "true", "yes"}
FA2_RELEASE_SCAN_LIMIT = int(os.getenv("FA2_RELEASE_SCAN_LIMIT", "30"))
WHEEL_REPO = "mjun0812/flash-attention-prebuild-wheels"
FA2_PREFERRED_RELEASES = [x.strip() for x in os.getenv(
    "FA2_PREFERRED_RELEASES", "v0.9.4,v0.9.44,v0.9.52").split(",") if x.strip()]
CLOUDFLARED_DEB_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"

# Qwen3: transformers>=4.51.
PIP_SPECS = [
    "packaging",
    "transformers>=4.51.0",
    "accelerate>=1.0.0",
    "peft>=0.14.0",
    "tokenizers>=0.21.0",
    "sentence-transformers>=3.0.0",
    "qdrant-client>=1.9.0",
    "fastapi>=0.110.0",
    "uvicorn>=0.27.0",
    "httpx>=0.27.0",
    "pydantic>=2.0.0",
    "nest-asyncio>=1.5.0",
    "captum>=0.7.0",
    "datasets>=2.18.0",
    "kiwipiepy>=0.18.0",
    "openai>=1.40.0",
]


def run_pip_logged(arguments, label):
    result = subprocess.run([sys.executable, "-m", "pip", *arguments],
                            capture_output=True, text=True)
    path = INSTALL_LOG_DIR / f"{INSTALL_RUN_ID}_{label}.log"
    path.write_text(result.stdout + "\n" + result.stderr, encoding="utf-8")
    if result.returncode:
        raise RuntimeError(f"{label} 실패: {path}\n{result.stderr[-3000:]}")


loaded_libraries = [name for name in ("transformers", "peft", "captum", "sentence_transformers", "tokenizers") if name in sys.modules]
if loaded_libraries:
    raise RuntimeError(f"이미 모델 라이브러리가 로드되어 있습니다: {loaded_libraries}. 설치 셀은 새 런타임에서 실행해 주세요.")

constraint_path = INSTALL_LOG_DIR / f"{INSTALL_RUN_ID}_torch_constraint.txt"
constraints = [f"torch=={torch.__version__}"]
try:
    constraints.append("numpy==" + importlib.metadata.version("numpy"))
except importlib.metadata.PackageNotFoundError:
    pass
constraint_path.write_text("\n".join(constraints) + "\n", encoding="utf-8")
run_pip_logged(["install", "-q", "-U", "-c", str(constraint_path), *PIP_SPECS], "dependencies")
if importlib.metadata.version("torch") != torch.__version__:
    raise RuntimeError("실행 중 torch와 설치된 torch가 다릅니다. 런타임을 재시작해 주세요.")

from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename
from packaging.version import Version


def verify_critical_packages():
    minimum_versions = {
        "transformers": "4.51.0",
        "peft": "0.14.0",
        "httpx": "0.27.0",
    }
    report = {}
    for name, minimum in minimum_versions.items():
        version = Version(importlib.metadata.version(name))
        report[name] = str(version)
        if version < Version(minimum):
            raise RuntimeError(f"{name} {version} < {minimum}")
    return report


PACKAGE_VERSIONS = verify_critical_packages()
print("[OK] 핵심 패키지:", PACKAGE_VERSIONS)


def peft_compatibility_probe():
    code = """
import torch
from transformers import Qwen3Config, Qwen3ForCausalLM
from peft import LoraConfig, get_peft_model
config = Qwen3Config(vocab_size=32, hidden_size=32, intermediate_size=64, num_hidden_layers=1,
                    num_attention_heads=4, num_key_value_heads=2, head_dim=8)
model = get_peft_model(Qwen3ForCausalLM(config), LoraConfig(r=2, lora_alpha=4,
                       target_modules=['q_proj', 'v_proj'], task_type='CAUSAL_LM'))
model.eval()
with torch.no_grad():
    output = model(input_ids=torch.tensor([[1, 2, 3]]), use_cache=False).logits
assert output.shape == (1, 3, 32) and torch.isfinite(output).all()
print('PEFT_QWEN_SMOKE_OK')
"""
    result = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=180)
    path = INSTALL_LOG_DIR / f"{INSTALL_RUN_ID}_peft_probe.log"
    with path.open("a", encoding="utf-8") as stream:
        stream.write(result.stdout + "\n" + result.stderr + "\n")
    return result.returncode == 0 and "PEFT_QWEN_SMOKE_OK" in result.stdout, result.stderr


def ensure_torchao_for_peft():
    passed, stderr = peft_compatibility_probe()
    if passed:
        return {"passed": True, "torchao_action": "unchanged"}
    actions = []
    lowered = stderr.lower()
    if "qwen3" in lowered:
        run_pip_logged(["install", "-q", "-U", "-c", str(constraint_path), "transformers>=4.51.0"], "transformers_upgrade")
        importlib.invalidate_caches()
        actions.append("transformers_upgraded")
        passed, stderr = peft_compatibility_probe()
        if passed:
            return {"passed": True, "torchao_action": "+".join(actions)}
    if importlib.util.find_spec("torchao") is not None:
        try:
            if Version(importlib.metadata.version("torchao")) < Version("0.16.0"):
                run_pip_logged(["install", "-q", "-U", "-c", str(constraint_path), "torchao>=0.16.0"], "torchao_upgrade")
                importlib.invalidate_caches()
                actions.append("torchao_upgraded")
                passed, stderr = peft_compatibility_probe()
                if passed:
                    return {"passed": True, "torchao_action": "+".join(actions)}
        except importlib.metadata.PackageNotFoundError:
            pass
    policy = os.getenv("TORCHAO_POLICY", "remove_if_broken")
    installed = importlib.util.find_spec("torchao") is not None
    if policy != "remove_if_broken" or not installed or "torchao" not in stderr.lower():
        raise RuntimeError("Qwen/PEFT 호환 시험 실패. peft_probe.log를 확인해 주세요.\n" + stderr[-2000:])
    run_pip_logged(["uninstall", "-y", "torchao"], "torchao_remove")
    importlib.invalidate_caches()
    passed, stderr = peft_compatibility_probe()
    if not passed:
        raise RuntimeError("torchao 제거 후에도 PEFT 검증 실패. peft_probe.log를 확인해 주세요.\n" + stderr[-2000:])
    actions.append("torchao_removed")
    print("[OK] Qwen/LoRA forward 검증 완료")
    return {"passed": True, "torchao_action": "+".join(actions)}


PEFT_COMPATIBILITY = ensure_torchao_for_peft()


def ensure_cloudflared():
    try:
        probe = subprocess.run(["cloudflared", "--version"], capture_output=True, text=True)
    except FileNotFoundError:
        probe = None
    if probe is not None and probe.returncode == 0:
        line = (probe.stdout or probe.stderr).strip().splitlines()[0]
        print("[OK] cloudflared:", line)
        return {"installed": True, "method": "existing", "version_line": line}
    deb_path = Path("/content") / "cloudflared.deb"
    print("[INFO] cloudflared 설치 중...")
    with urllib.request.urlopen(CLOUDFLARED_DEB_URL, timeout=60) as response, deb_path.open("wb") as output:
        while chunk := response.read(1024 * 1024):
            output.write(chunk)
    install = subprocess.run(["dpkg", "-i", str(deb_path)], capture_output=True, text=True)
    log_path = INSTALL_LOG_DIR / f"{INSTALL_RUN_ID}_cloudflared.log"
    log_path.write_text((install.stdout or "") + "\n" + (install.stderr or ""), encoding="utf-8")
    try:
        probe = subprocess.run(["cloudflared", "--version"], capture_output=True, text=True)
    except FileNotFoundError as exc:
        raise RuntimeError(f"cloudflared 설치 후 실행 파일 없음: {log_path}") from exc
    if probe.returncode != 0:
        raise RuntimeError(f"cloudflared 설치 실패: {log_path}\n{probe.stderr[-1000:]}")
    line = (probe.stdout or probe.stderr).strip().splitlines()[0]
    print("[OK] cloudflared 설치:", line)
    return {"installed": True, "method": "dpkg", "version_line": line}


CLOUDFLARED_REPORT = ensure_cloudflared()


def runtime_wheel_keys(torch_version, cuda_version):
    torch_pair = ".".join(str(torch_version).split("+")[0].split(".")[:2])
    cuda_tag = str(cuda_version or "").replace(".", "")
    python_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    return torch_pair, cuda_tag, python_tag


def compatible_fa2_wheel(filename, torch_version, cuda_version, supported_tags):
    try:
        name, version, _, tags = parse_wheel_filename(filename)
    except Exception:
        return False
    if name.replace("-", "_") != "flash_attn" or version.major != 2:
        return False
    match = re.search(r"\+cu(\d+)torch(\d+\.\d+)(?=-)", filename)
    torch_pair = ".".join(str(torch_version).split("+")[0].split(".")[:2])
    cuda_tag = str(cuda_version or "").replace(".", "")
    return bool(match and match.group(1) == cuda_tag and match.group(2) == torch_pair
                and tags.intersection(set(supported_tags)))


def fa2_match_score(filename, torch_version, cuda_version):
    if not compatible_fa2_wheel(filename, torch_version, cuda_version, set(sys_tags())):
        return None
    torch_pair, cuda_tag, python_tag = runtime_wheel_keys(torch_version, cuda_version)
    wheel_version = parse_wheel_filename(filename)[1]
    score = wheel_version.major * 1_000_000 + wheel_version.minor * 1_000 + wheel_version.micro
    if f"{python_tag}-{python_tag}" in filename:
        score += 100_000
    if f"torch{torch_pair}" in filename:
        score += 10_000
    if f"cu{cuda_tag}" in filename:
        score += 1_000
    if "linux_x86_64" in filename:
        score += 100
    return score


def summarize_wheel_candidates(assets, torch_version, cuda_version, supported_tags, limit=5):
    torch_pair, cuda_tag, python_tag = runtime_wheel_keys(torch_version, cuda_version)
    compatible = [
        a["name"] for a in assets
        if fa2_match_score(a["name"], torch_version, cuda_version) is not None
    ]
    same_python = [n for n in compatible if f"{python_tag}-{python_tag}" in n]
    same_torch = [n for n in same_python if f"torch{torch_pair}" in n]
    same_cuda = [n for n in same_torch if f"cu{cuda_tag}" in n]
    return {
        "runtime": {"python": python_tag, "torch": torch_pair, "cuda": cuda_tag},
        "compatible_assets": len(compatible),
        "same_python": same_python[:limit],
        "same_torch": same_torch[:limit],
        "same_cuda": same_cuda[:limit],
    }


def read_public_json(url):
    request = urllib.request.Request(url, headers={"User-Agent": "Capstone-FA-wheel-check",
                                                  "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.load(response)


def fetch_release_assets(tag):
    release = read_public_json(f"https://api.github.com/repos/{WHEEL_REPO}/releases/tags/{tag}")
    assets = []
    page = 1
    while True:
        batch = read_public_json(f"{release['assets_url']}?per_page=100&page={page}")
        assets.extend(batch)
        if len(batch) < 100:
            break
        page += 1
    return assets


def choose_fa2_asset(assets, torch_version, cuda_version, supported_tags):
    ranked = []
    for asset in assets:
        score = fa2_match_score(asset["name"], torch_version, cuda_version)
        if score is not None:
            ranked.append((score, asset))
    if not ranked:
        hint = summarize_wheel_candidates(assets, torch_version, cuda_version, supported_tags)
        raise RuntimeError(
            "일치하는 FA2 wheel을 찾지 못했습니다. "
            f"runtime={hint['runtime']}, same_python={hint['same_python']}, "
            f"same_torch={hint['same_torch']}, same_cuda={hint['same_cuda']}"
        )
    return max(ranked, key=lambda item: item[0])[1]


def candidate_fa2_filenames():
    torch_pair, cuda_tag, python_tag = runtime_wheel_keys(torch.__version__, torch.version.cuda)
    suffixes = [
        f"-{python_tag}-{python_tag}-linux_x86_64.whl",
        f"-{python_tag}-{python_tag}-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl",
    ]
    prefixes = ["flash_attn-2.8.3", "flash_attn-2.8.2", "flash_attn-2.8.1"]
    names = []
    for prefix in prefixes:
        for suffix in suffixes:
            names.append(f"{prefix}+cu{cuda_tag}torch{torch_pair}{suffix}")
    return names


def probe_release_for_runtime(tag):
    assets = fetch_release_assets(tag)
    return choose_fa2_asset(assets, torch.__version__, torch.version.cuda, set(sys_tags()))


def built_in_fa2_asset():
    torch_pair, cuda_tag, python_tag = runtime_wheel_keys(torch.__version__, torch.version.cuda)
    for tag in FA2_PREFERRED_RELEASES:
        for name in candidate_fa2_filenames():
            if not compatible_fa2_wheel(name, torch.__version__, torch.version.cuda, set(sys_tags())):
                continue
            url = f"https://github.com/{WHEEL_REPO}/releases/download/{tag}/{name}"
            try:
                request = urllib.request.Request(url, method="HEAD")
                with urllib.request.urlopen(request, timeout=20):
                    return {"name": name, "browser_download_url": url, "release": tag}
            except urllib.error.HTTPError:
                continue
    raise RuntimeError(
        f"선호 릴리스({FA2_PREFERRED_RELEASES})에 현재 런타임 wheel이 없습니다. "
        f"python={python_tag}, torch={torch_pair}, cuda={cuda_tag}"
    )


def auto_find_fa2_asset():
    tags = list(FA2_PREFERRED_RELEASES)
    releases = read_public_json(
        f"https://api.github.com/repos/{WHEEL_REPO}/releases?per_page={FA2_RELEASE_SCAN_LIMIT}")
    for release in releases:
        tag = release.get("tag_name")
        if tag and tag not in tags:
            tags.append(tag)
    all_assets = []
    for tag in tags:
        try:
            all_assets.extend(fetch_release_assets(tag))
        except Exception:
            continue
    if not all_assets:
        raise RuntimeError("FA2 wheel 릴리스 목록을 가져오지 못했습니다.")
    return choose_fa2_asset(all_assets, torch.__version__, torch.version.cuda, set(sys_tags()))


def resolve_fa2_asset():
    override = os.getenv("FLASH_ATTN_WHEEL_URL", "").strip()
    if override:
        parsed = urllib.parse.urlparse(override)
        name = urllib.parse.unquote(parsed.path.rsplit("/", 1)[-1])
        if parsed.scheme != "https" or parsed.hostname != "github.com" or not parsed.path.startswith(f"/{WHEEL_REPO}/releases/download/"):
            raise ValueError("기존 wheel 배포 저장소의 HTTPS release URL을 입력해 주세요.")
        return choose_fa2_asset([{"name": name, "browser_download_url": override}],
                               torch.__version__, torch.version.cuda, set(sys_tags()))
    try:
        return built_in_fa2_asset()
    except (RuntimeError, urllib.error.HTTPError) as exc:
        if not FA2_ENABLE_AUTO_SEARCH:
            raise RuntimeError(
                "현재 Colab 런타임용 FA2 wheel을 찾지 못했습니다. "
                "FA2_ENABLE_AUTO_SEARCH=1 또는 FLASH_ATTN_WHEEL_URL을 설정하세요."
            ) from exc
        print(f"[WARN] 내장 wheel 후보 없음 — 릴리스 자동 탐색: {exc}")
    return auto_find_fa2_asset()


RUNTIME_KEYS = runtime_wheel_keys(torch.__version__, torch.version.cuda)
FA_INSTALL_REPORT = {
    "environment": BOOT_ENV,
    "runtime_keys": {"python_tag": RUNTIME_KEYS[2], "torch_pair": RUNTIME_KEYS[0], "cuda_tag": RUNTIME_KEYS[1]},
    "require_flash_attn": REQUIRE_FLASH_ATTN,
    "package_versions": PACKAGE_VERSIONS,
    "cloudflared": CLOUDFLARED_REPORT,
    "wheel": None,
    "fa2": {"installed": False, "skipped": True},
    "peft_compatibility": PEFT_COMPATIBILITY,
}
if not REQUIRE_FLASH_ATTN:
    FA_INSTALL_REPORT["fa2"] = {"installed": False, "skipped": True}
    FA_INSTALL_REPORT["packages"] = {
        dist.metadata["Name"]: dist.version
        for dist in importlib.metadata.distributions() if dist.metadata["Name"]
    }
    report_path = INSTALL_LOG_DIR / f"{INSTALL_RUN_ID}_environment.json"
    FA_INSTALL_REPORT["report_path"] = str(report_path)
    report_path.write_text(json.dumps(FA_INSTALL_REPORT, ensure_ascii=False, indent=2), encoding="utf-8")
    print("환경/설치 결과:", report_path)
    print("[SKIP] FA2 wheel — REQUIRE_FLASH_ATTN=0 (vLLM + SDPA XAI)")
else:
    try:
        asset = resolve_fa2_asset()
        print("[INFO] FA2 wheel:", asset["browser_download_url"])
        FA_INSTALL_REPORT["wheel"] = asset["browser_download_url"]
        wheel_path = Path("/content") / asset["name"]
        with urllib.request.urlopen(asset["browser_download_url"], timeout=60) as response, wheel_path.open("wb") as output:
            while chunk := response.read(1024 * 1024):
                output.write(chunk)
        with wheel_path.open("rb") as stream:
            sha = hashlib.file_digest(stream, "sha256").hexdigest()
        FA_INSTALL_REPORT["sha256"] = sha
        if "flash_attn" in sys.modules:
            installed = importlib.metadata.version("flash-attn")
            if Version(installed) != parse_wheel_filename(asset["name"])[1]:
                raise RuntimeError("다른 flash-attn이 이미 로드되었습니다. 새 런타임에서 실행해 주세요.")
        else:
            run_pip_logged(["install", "--no-deps", "--force-reinstall", str(wheel_path)], "flash_attn")
            importlib.invalidate_caches()
        FA_INSTALL_REPORT["fa2"] = {"installed": True, "skipped": True}
    except Exception as exc:
        FA_INSTALL_REPORT["error"] = f"{type(exc).__name__}: {exc}"
        if REQUIRE_FLASH_ATTN:
            raise
        print("[WARN] FA2 검증 실패. SDPA fallback:", exc)
    finally:
        FA_INSTALL_REPORT["packages"] = {dist.metadata["Name"]: dist.version
                                         for dist in importlib.metadata.distributions() if dist.metadata["Name"]}
        report_path = INSTALL_LOG_DIR / f"{INSTALL_RUN_ID}_environment.json"
        FA_INSTALL_REPORT["report_path"] = str(report_path)
        report_path.write_text(json.dumps(FA_INSTALL_REPORT, ensure_ascii=False, indent=2), encoding="utf-8")
        print("환경/설치 결과:", report_path)



[OK] 핵심 패키지: {'transformers': '5.17.0', 'peft': '0.21.1', 'httpx': '0.28.1'}
[OK] cloudflared: cloudflared version 2026.9.3 (built 2026-09-24-16:07 UTC)
환경/설치 결과: /content/drive/MyDrive/Shared_Project/logs/xai_chatbot/20260930T003616947019Z_environment.json
[SKIP] FA2 wheel — REQUIRE_FLASH_ATTN=0 (vLLM + SDPA XAI)


In [4]:
# ============================================================
# [셀 05/26] 설정값 로드 및 기본 설정
# ============================================================
# 이 셀은 모델 ID, Qdrant 접속 정보, RAG 검색 개수,
# Reranker 모델 ID 등을 한 곳에서 관리한다.
#
# [설계 의도]
# - API KEY 같은 민감정보는 코드에 직접 쓰지 않고 Colab userdata에서 읽는다.
# - 모델/검색 개수는 실험 중 자주 바뀔 수 있으므로 설정값으로 분리한다.
# - 교수님/팀원 공유 시 어떤 외부 모델을 사용했는지 명확히 확인할 수 있도록 한다.
# ============================================================

import os
from google.colab import userdata


# ------------------------------------------------------------
# Colab userdata → 환경변수 → 기본값 순서로 설정값을 읽는 함수
# ------------------------------------------------------------
def get_setting(key: str, default: str = "") -> str:
    """
    설정값을 읽는 공통 함수.

    우선순위:
    1. Colab userdata
    2. 환경변수(os.environ)
    3. 코드에 지정한 기본값
    """
    try:
        value = userdata.get(key)
        if value is not None and str(value).strip():
            return str(value).strip()
    except Exception:
        pass

    value = os.getenv(key)
    if value is not None and str(value).strip():
        return str(value).strip()

    return default

# ============================================================
# Hugging Face Token 설정
# ============================================================

from huggingface_hub import login

HF_TOKEN = get_setting("HF_TOKEN", "")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face token 설정 완료")
else:
    print("[WARN] HF_TOKEN이 설정되지 않았습니다. 공개 모델은 다운로드 가능하지만 rate limit 경고가 발생할 수 있습니다.")


def get_int_setting(key: str, default: int) -> int:
    """
    정수형 설정값을 읽는 함수.
    예: QDRANT_FETCH_K, RERANK_TOP_N, FINAL_TOP_K
    """
    value = get_setting(key, str(default))
    try:
        return int(value)
    except ValueError:
        print(f"[WARN] {key}={value} 값을 int로 변환할 수 없어 기본값 {default} 사용")
        return default


def get_float_setting(key: str, default: float) -> float:
    """
    실수형 설정값을 읽는 함수.
    예: GEN_TEMPERATURE, GEN_TOP_P
    """
    value = get_setting(key, str(default))
    try:
        return float(value)
    except ValueError:
        print(f"[WARN] {key}={value} 값을 float로 변환할 수 없어 기본값 {default} 사용")
        return default


# ============================================================
#  LLM 설정
# ============================================================

MODEL_ID = get_setting("MODEL_ID", "Qwen/Qwen3-4B-Instruct-2507")


# ============================================================
# Embedding 모델 설정
# ============================================================
# - BAAI/bge-m3 사용
# - 역할: 사용자 질문을 벡터로 변환하여 Qdrant dense retrieval 수행
# - bge-m3는 다국어 검색에 활용 가능한 embedding 모델
# ============================================================

EMBED_MODEL_ID = get_setting("EMBED_MODEL_ID", "BAAI/bge-m3")


# ============================================================
# [외부 공개 모델 활용] Korean Reranker 설정
# ============================================================
# - Dongjin-kr/ko-reranker 사용
# - 역할: Qdrant 1차 검색 후보를 질문-문서 관련성 기준으로 재정렬
# - MIT License로 공개된 한국어 reranker
#
# [RAG 설계 근거]
# - 1차 dense retrieval은 후보 문서를 넓게 가져오는 recall 확보 단계
# - 2차 reranking은 질문과 문서를 pair로 비교하여 precision을 높이는 단계
# ============================================================

RERANKER_MODEL_ID = get_setting("RERANKER_MODEL_ID", "Dongjin-kr/ko-reranker")


# ============================================================
#  Qdrant 설정
# ============================================================

QDRANT_URL = get_setting("QDRANT_URL", "")
QDRANT_API_KEY = get_setting("QDRANT_API_KEY", "")

# 여러 컬렉션을 쉼표로 입력 가능
# 예: welfare_ministry_bge_m3,local_government_welfare_bge_m3
QDRANT_COLLECTIONS_RAW = get_setting(
    "QDRANT_COLLECTIONS",
    "welfare_policies_bge_m3"
)

QDRANT_COLLECTIONS = [
    name.strip()
    for name in QDRANT_COLLECTIONS_RAW.split(",")
    if name.strip()
]

# named vector를 쓰는 경우에만 설정
# 일반 단일 벡터 컬렉션이면 빈 값으로 둔다.
QDRANT_VECTOR_NAME = get_setting("QDRANT_VECTOR_NAME", "").strip() or None

# ============================================================
# RAG 검색 설정
# ============================================================
# - QDRANT_FETCH_K: 1차 dense retrieval 후보 수
# - RERANK_TOP_N: reranker 재정렬 후 남길 후보 수
# - FINAL_TOP_K: LLM context에 최종 투입할 정책 수
#
# 기본 전략:
# 1. Qdrant에서 후보를 넉넉히 가져온다.  예: 30개
# 2. Reranker로 질문 관련성을 재평가한다. 예: 상위 10개
# 3. 중복 제거 후 최종 3~5개만 LLM에 넣는다.
# ============================================================

QDRANT_FETCH_K = get_int_setting("QDRANT_FETCH_K", 30)
RERANK_TOP_N = get_int_setting("RERANK_TOP_N", 10)
FINAL_TOP_K = get_int_setting("FINAL_TOP_K", 5)

# original은 정책별 900자 문맥 제한을 적용한다.
MAX_CONTEXT_CHARS_PER_POLICY = get_int_setting("MAX_CONTEXT_CHARS_PER_POLICY", 900)
# ============================================================
# 답변 생성 설정
# ============================================================

MAX_NEW_TOKENS = get_int_setting("MAX_NEW_TOKENS", 700)
GEN_TEMPERATURE = get_float_setting("GEN_TEMPERATURE", 0.3)
GEN_TOP_P = get_float_setting("GEN_TOP_P", 0.9)
GEN_REPETITION_PENALTY = get_float_setting("GEN_REPETITION_PENALTY", 1.1)

# 선택 검증: 1이면 셀 8에서 두 모델의 출력 일치를 확인한다.
VERIFY_MODEL_PARITY = get_setting("VERIFY_MODEL_PARITY", "0").strip().lower() in {"1", "true", "yes", "y", "on"}

SKIP_ANSWER_MODEL_LOAD = True
VLLM_PORT = get_int_setting("VLLM_PORT", 8001)
VLLM_GPU_MEMORY_UTILIZATION = get_float_setting("VLLM_GPU_MEMORY_UTILIZATION", 0.30)
VLLM_MAX_MODEL_LEN = get_int_setting("VLLM_MAX_MODEL_LEN", 8192)
TARGET_GPU_NAME = "NVIDIA RTX PRO 6000 Blackwell Server Edition"


# ============================================================
# 설정 확인 출력
# ============================================================

print("========== 설정 확인 ==========")
print(f"MODEL_ID              : {MODEL_ID}")
print(f"EMBED_MODEL_ID        : {EMBED_MODEL_ID}")
print(f"RERANKER_MODEL_ID     : {RERANKER_MODEL_ID}")
print(f"QDRANT_URL 설정 여부  : {'OK' if QDRANT_URL else 'MISSING'}")
print(f"QDRANT_API_KEY 여부   : {'OK' if QDRANT_API_KEY else 'MISSING'}")
print(f"QDRANT_COLLECTIONS    : {QDRANT_COLLECTIONS}")
print(f"QDRANT_VECTOR_NAME    : {QDRANT_VECTOR_NAME}")
print(f"QDRANT_FETCH_K        : {QDRANT_FETCH_K}")
print(f"RERANK_TOP_N          : {RERANK_TOP_N}")
print(f"FINAL_TOP_K           : {FINAL_TOP_K}")
print(f"MAX_NEW_TOKENS        : {MAX_NEW_TOKENS}")
print(f"VERIFY_MODEL_PARITY   : {VERIFY_MODEL_PARITY}")
print(f"SKIP_ANSWER_MODEL_LOAD: {SKIP_ANSWER_MODEL_LOAD}")
print(f"VLLM_PORT             : {VLLM_PORT}")
print(f"VLLM_GPU_MEM_UTIL     : {VLLM_GPU_MEMORY_UTILIZATION}")
print(f"VLLM_MAX_MODEL_LEN    : {VLLM_MAX_MODEL_LEN}")
print(f"TARGET_GPU_NAME       : {TARGET_GPU_NAME}")
print("================================")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face token 설정 완료
========== 설정 확인 ==========
MODEL_ID              : Qwen/Qwen3-4B-Instruct-2507
EMBED_MODEL_ID        : BAAI/bge-m3
RERANKER_MODEL_ID     : Dongjin-kr/ko-reranker
QDRANT_URL 설정 여부  : OK
QDRANT_API_KEY 여부   : OK
QDRANT_COLLECTIONS    : ['welfare_policies_bge_m3']
QDRANT_VECTOR_NAME    : None
QDRANT_FETCH_K        : 30
RERANK_TOP_N          : 10
FINAL_TOP_K           : 5
MAX_NEW_TOKENS        : 700
VERIFY_MODEL_PARITY   : False
SKIP_ANSWER_MODEL_LOAD: True
VLLM_PORT             : 8001
VLLM_GPU_MEM_UTIL     : 0.3
VLLM_MAX_MODEL_LEN    : 8192
TARGET_GPU_NAME       : NVIDIA RTX PRO 6000 Blackwell Server Edition


In [5]:
# ============================================================
# [셀 06/26] 필수 설정 검증 및 로컬 경로 설정
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# 필수 설정 검증
# ------------------------------------------------------------
missing_settings = [
    name
    for name, value in (
        ("QDRANT_URL", QDRANT_URL),
        ("QDRANT_API_KEY", QDRANT_API_KEY),
        ("QDRANT_COLLECTIONS", QDRANT_COLLECTIONS),
    )
    if not value
]

if missing_settings:
    raise RuntimeError(
        "필수 설정이 누락되었습니다: "
        + ", ".join(missing_settings)
        + "\nColab userdata 또는 환경변수에 값을 설정해 주세요."
    )


# ------------------------------------------------------------
# 경로 설정
# ------------------------------------------------------------

RAG_LOG_PATH = LOG_DIR / "rag_debug.log"

LOCAL_LLM_PATH = LOCAL_MODEL_DIR / "qwen3-4b-instruct-2507"
ADAPTER_PATH = LOCAL_MODEL_DIR / "qwen3-4b-instruct-2507-adapter"

LOCAL_EMBED_PATH = LOCAL_MODEL_DIR / "bge-m3"
LOCAL_RERANKER_PATH = LOCAL_MODEL_DIR / "ko-reranker"

# 경로 생성
for directory in (LOCAL_LLM_PATH, LOCAL_EMBED_PATH, LOCAL_RERANKER_PATH, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 설정 확인 출력
# ------------------------------------------------------------
print("========== 4단계 설정 검증 완료 ==========")
print("Qdrant URL         :", "OK")
print("Qdrant API Key     :", "OK")
print("Qdrant Collections :", QDRANT_COLLECTIONS)
# print("LLM local path     :", LOCAL_LLM_PATH)
# print("Embed local path   :", LOCAL_EMBED_PATH)
# print("Reranker path      :", LOCAL_RERANKER_PATH)
print("RAG log path       :", RAG_LOG_PATH)
print("==========================================")


========== 4단계 설정 검증 완료 ==========
Qdrant URL         : OK
Qdrant API Key     : OK
Qdrant Collections : ['welfare_policies_bge_m3']
RAG log path       : /content/drive/MyDrive/Shared_Project/logs/rag_debug.log


In [6]:
# [셀 07/26] LLM 로컬 경로 확인 (선택)
import os

print(LOCAL_LLM_PATH)
if LOCAL_LLM_PATH.exists():
    files = os.listdir(LOCAL_LLM_PATH)
    if not files:
        print("비어있음")
    else:
        for filename in files:
            file_path = LOCAL_LLM_PATH / filename
            size_mb = os.path.getsize(file_path) / (1024 ** 2)
            print(f"- {filename} ({size_mb:.2f} MB)")
else:
    print("폴더 자체 존재 X")


/content/drive/MyDrive/Shared_Project/local_models/qwen3-4b-instruct-2507
- chat_template.jinja (0.00 MB)
- tokenizer_config.json (0.00 MB)
- tokenizer.json (10.89 MB)
- config.json (0.00 MB)
- generation_config.json (0.00 MB)
- model.safetensors (7672.29 MB)


In [7]:
# ============================================================
# [셀 08/26] XAI SDPA 모델 로드. 답변 HF 모델은 로드하지 않는다 (vLLM이 생성).
# - XAI/Captum: SDPA, use_cache=False
# - SKIP_ANSWER_MODEL_LOAD=True가 기본이다
# ============================================================

missing = [key for key in ("FA_INSTALL_REPORT", "REQUIRE_FLASH_ATTN", "LOCAL_LLM_PATH", "ADAPTER_PATH", "MODEL_ID", "VERIFY_MODEL_PARITY") if key not in globals()]
if missing:
    raise RuntimeError(f"셀 02~06을 순서대로 먼저 실행해 주세요. 누락: {missing}")

import gc
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

USE_HYBRID_ATTENTION = os.getenv("USE_HYBRID_ATTENTION", "1").strip().lower() in {"1", "true", "yes", "y", "on"}
MERGE_LORA_FOR_INFERENCE = os.getenv("MERGE_LORA_FOR_INFERENCE", "1").strip().lower() in {"1", "true", "yes", "y", "on"}
ANSWER_ATTENTION_BACKEND = os.getenv("ANSWER_ATTENTION_BACKEND", "flash_attention_2").strip() or "flash_attention_2"
XAI_ATTENTION_BACKEND = os.getenv("XAI_ATTENTION_BACKEND", "sdpa").strip() or "sdpa"

print("LLM device:", DEVICE)
print("LLM dtype :", TORCH_DTYPE)
print("Answer attention backend:", ANSWER_ATTENTION_BACKEND)
print("XAI attention backend:", XAI_ATTENTION_BACKEND)
print("Merge LoRA for inference:", MERGE_LORA_FOR_INFERENCE)


def has_local_hf_model_files(model_path: Path) -> bool:
    required_files = ["config.json", "tokenizer_config.json"]
    has_required = all((model_path / filename).exists() for filename in required_files)
    has_weight = (
        (model_path / "model.safetensors").exists()
        or (model_path / "model.safetensors.index.json").exists()
        or (model_path / "pytorch_model.bin").exists()
        or (model_path / "pytorch_model.bin.index.json").exists()
    )
    return has_required and has_weight


if has_local_hf_model_files(LOCAL_LLM_PATH):
    print(f"[INFO] loading local base LLM: {LOCAL_LLM_PATH}")
    llm_source = str(LOCAL_LLM_PATH)
    local_files_only = True
else:
    print(f"[INFO] local LLM files not found; loading from Hugging Face: {MODEL_ID}")
    llm_source = MODEL_ID
    local_files_only = False

# 답변 모델과 XAI 모델은 토크나이저를 공유한다.
tokenizer = AutoTokenizer.from_pretrained(
    llm_source,
    trust_remote_code=True,
    local_files_only=local_files_only,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def get_model_device(active_model):
    try:
        return next(active_model.parameters()).device
    except StopIteration:
        return torch.device(DEVICE)


def load_qwen_model_for_role(role: str, attention_backend: str, use_cache: bool):
    load_kwargs = {
        "torch_dtype": TORCH_DTYPE,
        "trust_remote_code": True,
        "local_files_only": local_files_only,
    }
    if attention_backend:
        load_kwargs["attn_implementation"] = attention_backend
    if DEVICE == "cuda":
        load_kwargs["device_map"] = {"": 0}

    selected_attention = attention_backend or "default"
    attention_error = None

    try:
        loaded_model = AutoModelForCausalLM.from_pretrained(llm_source, **load_kwargs)
    except Exception as exc:
        attention_error = f"{type(exc).__name__}: {exc}"
        if not REQUIRE_FLASH_ATTN and attention_backend and attention_backend != "sdpa":
            print(f"[WARN] {role} attention backend failed: {attention_backend}; falling back to SDPA.")
            print(f"[WARN] {attention_error}")
            load_kwargs["attn_implementation"] = "sdpa"
            selected_attention = "sdpa_fallback"
            loaded_model = AutoModelForCausalLM.from_pretrained(llm_source, **load_kwargs)
        else:
            raise

    if role == "xai" and not local_files_only:
        print(f"[INFO] saving base model and tokenizer: {LOCAL_LLM_PATH}")
        tokenizer.save_pretrained(str(LOCAL_LLM_PATH))
        loaded_model.save_pretrained(str(LOCAL_LLM_PATH))

    lora_status = "not_found"
    lora_error = None
    if ADAPTER_PATH.exists() and (ADAPTER_PATH / "adapter_config.json").exists():
        print(f"[INFO] loading LoRA adapter for {role}: {ADAPTER_PATH}")
        loaded_model = PeftModel.from_pretrained(loaded_model, str(ADAPTER_PATH))
        lora_status = "loaded"
        if MERGE_LORA_FOR_INFERENCE:
            if hasattr(loaded_model, "merge_and_unload"):
                try:
                    loaded_model = loaded_model.merge_and_unload()
                    lora_status = "merged_for_inference"
                except Exception as exc:
                    lora_error = f"{type(exc).__name__}: {exc}"
                    print(f"[WARN] {role} LoRA merge failed; keeping adapter attached: {lora_error}")
            else:
                lora_status = "loaded_merge_unavailable"
                print(f"[WARN] merge_and_unload is unavailable for {role}; keeping adapter attached.")
    else:
        print(f"[INFO] LoRA adapter not found for {role}; using base model.")

    loaded_model.eval()
    if hasattr(loaded_model, "config"):
        loaded_model.config.use_cache = bool(use_cache)
    if hasattr(loaded_model, "generation_config"):
        loaded_model.generation_config.use_cache = bool(use_cache)

    return loaded_model, selected_attention, lora_status, attention_error, lora_error


# 답변 생성은 vLLM이 담당한다.
SKIP_ANSWER_MODEL_LOAD = bool(globals().get("SKIP_ANSWER_MODEL_LOAD", True))

# Captum은 전역 변수 `model`을 참조하므로 XAI용 모델을 `model`에 둔다.
model, XAI_ATTENTION_DETAIL, XAI_LORA_STATUS, XAI_ATTENTION_ERROR, XAI_LORA_ERROR = load_qwen_model_for_role(
    role="xai",
    attention_backend=XAI_ATTENTION_BACKEND,
    use_cache=False,
)

HYBRID_RUNTIME_SHARED_MODEL = not USE_HYBRID_ATTENTION

if SKIP_ANSWER_MODEL_LOAD:
    print("[INFO] SKIP_ANSWER_MODEL_LOAD=True — vLLM이 답변 생성을 대신하므로 답변용 HF 모델 로드를 건너뜁니다.")
    generation_model = None
    ANSWER_ATTENTION_DETAIL = "skipped_vllm"
    ANSWER_LORA_STATUS = XAI_LORA_STATUS  # 아래 상태 비교 검증을 통과시키기 위해 XAI와 동일하게 둠
    ANSWER_ATTENTION_ERROR = None
    ANSWER_LORA_ERROR = None
    HYBRID_RUNTIME_SHARED_MODEL = False
elif USE_HYBRID_ATTENTION:
    try:
        generation_model, ANSWER_ATTENTION_DETAIL, ANSWER_LORA_STATUS, ANSWER_ATTENTION_ERROR, ANSWER_LORA_ERROR = load_qwen_model_for_role(
            role="answer",
            attention_backend=ANSWER_ATTENTION_BACKEND,
            use_cache=True,
        )
    except Exception as exc:
        if REQUIRE_FLASH_ATTN:
            raise
        print(f"[WARN] answer model load failed; using the XAI model for generation: {type(exc).__name__}: {exc}")
        generation_model = model
        ANSWER_ATTENTION_DETAIL = XAI_ATTENTION_DETAIL
        ANSWER_LORA_STATUS = XAI_LORA_STATUS
        ANSWER_ATTENTION_ERROR = f"answer_model_load_failed: {type(exc).__name__}: {exc}"
        ANSWER_LORA_ERROR = XAI_LORA_ERROR
        HYBRID_RUNTIME_SHARED_MODEL = True
else:
    generation_model = model
    ANSWER_ATTENTION_DETAIL = XAI_ATTENTION_DETAIL
    ANSWER_LORA_STATUS = XAI_LORA_STATUS
    ANSWER_ATTENTION_ERROR = XAI_ATTENTION_ERROR
    ANSWER_LORA_ERROR = XAI_LORA_ERROR

XAI_INFERENCE_DEVICE = get_model_device(model)
ANSWER_INFERENCE_DEVICE = (
    get_model_device(generation_model) if generation_model is not None else "vllm(external)"
)

LLM_RUNTIME_CONFIG = {
    "hybrid_attention": USE_HYBRID_ATTENTION,
    "answer_attention_backend": ANSWER_ATTENTION_DETAIL,
    "xai_attention_backend": XAI_ATTENTION_DETAIL,
    "answer_lora_status": ANSWER_LORA_STATUS,
    "xai_lora_status": XAI_LORA_STATUS,
    "answer_use_cache": bool(getattr(getattr(generation_model, "config", None), "use_cache", False)),
    "xai_use_cache": bool(getattr(getattr(model, "config", None), "use_cache", False)),
    "answer_device": str(ANSWER_INFERENCE_DEVICE),
    "xai_device": str(XAI_INFERENCE_DEVICE),
    "hybrid_runtime_shared_model": HYBRID_RUNTIME_SHARED_MODEL,
}

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("========== LLM 로드 완료 ==========")
print("MODEL_ID              :", MODEL_ID)
print("cache_dir             :", HF_HUB_CACHE)
print("device                :", DEVICE)
print("dtype                 :", TORCH_DTYPE)
print("pad_token             :", tokenizer.pad_token)
print("eos_token_id          :", tokenizer.eos_token_id)
print("answer_attention      :", ANSWER_ATTENTION_DETAIL)
print("xai_attention         :", XAI_ATTENTION_DETAIL)
print("answer_lora           :", ANSWER_LORA_STATUS)
print("xai_lora              :", XAI_LORA_STATUS)
print("answer_use_cache      :", LLM_RUNTIME_CONFIG["answer_use_cache"])
print("xai_use_cache         :", LLM_RUNTIME_CONFIG["xai_use_cache"])
print("answer_device         :", ANSWER_INFERENCE_DEVICE)
print("xai_device            :", XAI_INFERENCE_DEVICE)
print("=======================================")

# 실제 HF backend와 두 역할의 어댑터 상태를 확인한다.
if REQUIRE_FLASH_ATTN and not SKIP_ANSWER_MODEL_LOAD:
    if getattr(generation_model.config, "_attn_implementation", None) != "flash_attention_2":
        raise RuntimeError("답변 모델이 flash_attention_2로 로드되지 않았습니다.")
if ANSWER_LORA_STATUS != XAI_LORA_STATUS:
    raise RuntimeError("답변 모델과 XAI 모델의 LoRA 적용 상태가 다릅니다.")
if getattr(model.config, "_attn_implementation", None) != "sdpa":
    raise RuntimeError("XAI 모델은 sdpa로 로드해 주세요.")

LLM_RUNTIME_CONFIG["actual_answer_backend"] = (
    generation_model.config._attn_implementation if generation_model is not None else "vllm(external)"
)
LLM_RUNTIME_CONFIG["actual_xai_backend"] = model.config._attn_implementation
LLM_RUNTIME_CONFIG["model_revision"] = getattr(model.config, "_commit_hash", None)
LLM_RUNTIME_CONFIG["probe_max_abs_logit_error"] = None

# 같은 가중치의 서로 다른 attention 경로가 같은 입력에서 수치적으로 일치하는지 확인한다.
if SKIP_ANSWER_MODEL_LOAD:
    print("두 모델 출력 일치 검증: 생략 (SKIP_ANSWER_MODEL_LOAD=True, 답변용 모델 없음)")
elif VERIFY_MODEL_PARITY:
    probe = tokenizer("정책의 지원 조건을 확인합니다.", return_tensors="pt").to(ANSWER_INFERENCE_DEVICE)
    answer_probe = xai_probe = None
    try:
        with torch.no_grad():
            answer_probe = generation_model(**probe, use_cache=False).logits[:, -1, :].float()
            xai_probe = model(**probe, use_cache=False).logits[:, -1, :].float()
            parity_max_error = float((answer_probe - xai_probe).abs().max())
            torch.testing.assert_close(answer_probe, xai_probe, atol=0.15, rtol=0.03)
        LLM_RUNTIME_CONFIG["probe_max_abs_logit_error"] = parity_max_error
        print("두 모델 출력 일치 검증:", parity_max_error)
    finally:
        del probe, answer_probe, xai_probe
else:
    print("두 모델 출력 일치 검증: 생략 (VERIFY_MODEL_PARITY=0)")


LLM device: cuda
LLM dtype : torch.bfloat16
Answer attention backend: flash_attention_2
XAI attention backend: sdpa
Merge LoRA for inference: True
[INFO] loading local base LLM: /content/drive/MyDrive/Shared_Project/local_models/qwen3-4b-instruct-2507


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[INFO] loading LoRA adapter for xai: /content/drive/MyDrive/Shared_Project/local_models/qwen3-4b-instruct-2507-adapter
[INFO] SKIP_ANSWER_MODEL_LOAD=True — vLLM이 답변 생성을 대신하므로 답변용 HF 모델 로드를 건너뜁니다.
========== LLM 로드 완료 ==========
MODEL_ID              : Qwen/Qwen3-4B-Instruct-2507
cache_dir             : /content/drive/MyDrive/Shared_Project/hf_cache/hub
device                : cuda
dtype                 : torch.bfloat16
pad_token             : <|endoftext|>
eos_token_id          : 151645
answer_attention      : skipped_vllm
xai_attention         : sdpa
answer_lora           : merged_for_inference
xai_lora              : merged_for_inference
answer_use_cache      : False
xai_use_cache         : False
answer_device         : vllm(external)
xai_device            : cuda:0
두 모델 출력 일치 검증: 생략 (SKIP_ANSWER_MODEL_LOAD=True, 답변용 모델 없음)


In [8]:
# [셀 09/26] 추론 검증 중 자동 재학습 방지
# 학습 후에는 런타임을 재시작하여 답변/분석 모델에 같은 어댑터를 다시 로드한다.
is_finetuned = ADAPTER_PATH.exists() and (ADAPTER_PATH / "adapter_config.json").exists()
print("저장된 LoRA 사용:" if is_finetuned else "기본 모델 사용:", ADAPTER_PATH)
print("이 노트북은 추론만 수행하며 자동 재학습하지 않습니다.")


저장된 LoRA 사용: /content/drive/MyDrive/Shared_Project/local_models/qwen3-4b-instruct-2507-adapter
이 노트북은 추론만 수행하며 자동 재학습하지 않습니다.


In [9]:
# ============================================================
# [셀 10/26] Embedding 모델 로드 - 기본값: BAAI/bge-m3
# ============================================================

from sentence_transformers import SentenceTransformer

print(f"[INFO] Embedding 모델 로드 시작: {EMBED_MODEL_ID}")

embed_model = SentenceTransformer(
    EMBED_MODEL_ID,
    device=DEVICE,
    cache_folder=str(LOCAL_EMBED_PATH)
)

print("========== Embedding 모델 로드 완료 ==========")
print("EMBED_MODEL_ID:", EMBED_MODEL_ID)
print("cache_dir     :", HF_HUB_CACHE)
print("device        :", DEVICE)
print("============================================")


[INFO] Embedding 모델 로드 시작: BAAI/bge-m3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

========== Embedding 모델 로드 완료 ==========
EMBED_MODEL_ID: BAAI/bge-m3
cache_dir     : /content/drive/MyDrive/Shared_Project/hf_cache/hub
device        : cuda


In [10]:
# ============================================================
# [셀 11/26] Qdrant 연결
# ============================================================
# 이 셀은 정책 문서 벡터가 저장된 Qdrant DB에 연결한다.
#
# - 역할: 사용자 질문 embedding과 유사한 정책 문서 chunk 검색
# - 중앙정부 정책 컬렉션과 지자체 정책 컬렉션을 함께 검색할 수 있도록
#   QDRANT_COLLECTIONS 목록을 기준으로 연결 상태를 확인한다.
# - 이 노트북은 검색만 수행하며, upsert/delete/recreate_collection은 사용하지 않는다.
# ============================================================

from qdrant_client import QdrantClient


# ------------------------------------------------------------
# Qdrant Client 생성
# ------------------------------------------------------------

print("[INFO] Qdrant 연결을 시작합니다.")

qdrant_client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

print("[INFO] Qdrant Client 생성 완료")


# ------------------------------------------------------------
# 검색 대상 컬렉션 확인
# ------------------------------------------------------------
# get_collection()은 컬렉션 존재 여부와 기본 정보를 확인하는 용도다.
# 검색 대상 컬렉션이 없거나 이름이 틀리면 여기서 미리 확인할 수 있다.
# ------------------------------------------------------------

print("========== Qdrant 컬렉션 확인 ==========")

collection_infos = {}
for collection_name in QDRANT_COLLECTIONS:
    try:
        info = qdrant_client.get_collection(collection_name)
        collection_infos[collection_name] = info
        print(f"[OK] {collection_name}")
        print(f"     vectors_count: {getattr(info, 'vectors_count', 'unknown')}")
        print(f"     points_count : {getattr(info, 'points_count', 'unknown')}")
    except Exception as e:
        print(f"[ERROR] 컬렉션 확인 실패: {collection_name}")
        print(f"        원인: {e}")
        raise

print("=======================================")
print("[INFO] Qdrant 연결 및 컬렉션 확인 완료")

# 연결 성공에 더해 질의 임베딩과 컬렉션의 벡터 구성을 확인한다.
QDRANT_SCHEMA_REPORT = {}
embedding_dimension = embed_model.get_sentence_embedding_dimension()
for collection_name in QDRANT_COLLECTIONS:
    collection_info = collection_infos[collection_name]
    vectors = collection_info.config.params.vectors
    if isinstance(vectors, dict):
        if not QDRANT_VECTOR_NAME or QDRANT_VECTOR_NAME not in vectors:
            raise ValueError("named vector 컬렉션의 QDRANT_VECTOR_NAME을 확인해 주세요.")
        vector_config = vectors[QDRANT_VECTOR_NAME]
    else:
        if QDRANT_VECTOR_NAME:
            raise ValueError("단일 unnamed vector 컬렉션에서는 QDRANT_VECTOR_NAME을 비워 주세요.")
        vector_config = vectors
    if vector_config.size != embedding_dimension:
        raise ValueError(f"{collection_name}: DB {vector_config.size}차원과 질의 {embedding_dimension}차원이 다릅니다.")
    QDRANT_SCHEMA_REPORT[collection_name] = {"points_count": collection_info.points_count,
        "vector_size": vector_config.size, "distance": str(vector_config.distance),
        "payload_indexes": sorted((collection_info.payload_schema or {}).keys())}
print("벡터 구성 검증:", QDRANT_SCHEMA_REPORT)


[INFO] Qdrant 연결을 시작합니다.
[INFO] Qdrant Client 생성 완료
========== Qdrant 컬렉션 확인 ==========
[OK] welfare_policies_bge_m3
     vectors_count: unknown
     points_count : 5219
[INFO] Qdrant 연결 및 컬렉션 확인 완료
벡터 구성 검증: {'welfare_policies_bge_m3': {'points_count': 5219, 'vector_size': 1024, 'distance': 'Cosine', 'payload_indexes': ['category', 'region_sido', 'region_sigungu', 'scope', 'service_id', 'source_types']}}


/tmp/ipykernel_29908/1504996022.py:56: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = embed_model.get_sentence_embedding_dimension()


In [11]:
# ============================================================
# [셀 12/26] Korean Reranker 모델 로드 - HF cache 통일 버전
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print(f"[INFO] Reranker 모델 로드 시작: {RERANKER_MODEL_ID}")

reranker_tokenizer = AutoTokenizer.from_pretrained(
    RERANKER_MODEL_ID,
    use_fast=True,
    trust_remote_code=True,
    cache_dir=str(LOCAL_RERANKER_PATH)  #드라이브 말고 colab 캐시에 저장하도록 명시
)

reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_MODEL_ID,
    torch_dtype=TORCH_DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True,
    cache_dir=str(LOCAL_RERANKER_PATH) #드라이브 말고 colab 캐시에 저장하도록 명시
)

reranker_model.eval()

print("========== Reranker 모델 로드 완료 ==========")
print("RERANKER_MODEL_ID:", RERANKER_MODEL_ID)
print("cache_dir        :", LOCAL_RERANKER_PATH)
print("device           :", DEVICE)
print("dtype            :", TORCH_DTYPE)
print("============================================")


[INFO] Reranker 모델 로드 시작: Dongjin-kr/ko-reranker


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

========== Reranker 모델 로드 완료 ==========
RERANKER_MODEL_ID: Dongjin-kr/ko-reranker
cache_dir        : /content/drive/MyDrive/Shared_Project/local_models/ko-reranker
device           : cuda
dtype            : torch.bfloat16


In [12]:
# ============================================================
# [셀 13/26] Qdrant payload 정제 함수
# ============================================================
# 이 셀은 Qdrant 검색 결과의 payload에서 정책명, 지원내용, 지역, URL 등을 추출한다.
#
# - Qdrant에 저장된 정책 문서의 payload 구조가 항상 동일하다고 가정하지 않는다.
# - payload 직접 key, metadata 내부 key, 본문 안의 [필드명] / 필드명: 형식을 모두 확인한다.
# - Reranker 입력과 LLM context에 사용할 수 있도록 정책 정보를 정리한다.
# ============================================================

import re
from typing import Any, Dict, List, Tuple


# ------------------------------------------------------------
# payload 값 추출
# ------------------------------------------------------------
# Qdrant payload는 저장 방식에 따라 아래 두 형태가 모두 가능하다.
#
# 1. payload["정책명"]
# 2. payload["metadata"]["정책명"]
#
# 따라서 직접 key와 metadata 내부 key를 모두 확인한다.
# ------------------------------------------------------------

def get_payload_value(payload: Dict[str, Any], *keys: str) -> str:
    """
    payload에서 여러 후보 key를 순서대로 확인해 값을 추출한다.
    직접 key와 metadata 내부 key를 모두 확인한다.
    """
    if not isinstance(payload, dict):
        return ""

    candidates = []

    # payload 직접 key 확인
    for key in keys:
        if key:
            candidates.append(payload.get(key))

    # metadata 내부 key 확인
    metadata = payload.get("metadata")
    if isinstance(metadata, dict):
        for key in keys:
            if key:
                candidates.append(metadata.get(key))

    for value in candidates:
        if value is None:
            continue
        text = str(value).strip()
        if text:
            return text

    return ""


# ------------------------------------------------------------
# 텍스트 정규화
# ------------------------------------------------------------

def normalize_policy_text(text: str) -> str:
    """
    모델과 reranker에 넣기 전 텍스트를 간단히 정리한다.
    """
    if not text:
        return ""

    text = str(text)
    text = text.replace("、", ", ").replace("。", ". ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_extracted_value(value: str) -> str:
    """
    필드 추출 결과에 다른 필드가 붙어 들어온 경우를 정리한다.
    예: '[지원내용] ... [신청방법] ...'에서 지원내용만 남기기
    """
    if not value:
        return ""

    value = normalize_policy_text(value)

    # 다음 [필드명]이 붙어 있으면 그 전까지만 사용
    value = re.split(
        r"\s+\[(?:서비스ID|서비스명|정책명|분류|분야|지역|지원유형|지원대상|지원내용|선정기준|신청정보|신청방법|상세 URL|URL|섹션)\]",
        value,
        maxsplit=1,
    )[0].strip()

    return value


# ------------------------------------------------------------
# 본문 안의 bracket 필드 추출
# ------------------------------------------------------------
# 예:
# [서비스명] 청년월세 지원사업
# [지원내용] 월세 일부 지원
# ------------------------------------------------------------

def extract_bracket_field(text: str, labels: List[str]) -> str:
    """
    [서비스명] ..., [지원내용] ... 같은 bracket 필드를 추출한다.
    다음 [필드명] 전까지만 가져온다.
    """
    if not text:
        return ""

    headers = "서비스ID|서비스명|정책명|분류|분야|지역|카테고리|지원유형|지원대상|지원 대상|지원내용|지원 내용|주요혜택|선정기준|연령조건|신청정보|신청방법|신청 방법|담당기관|상세 URL|URL|섹션"

    for label in labels:
        pattern = rf"\[{re.escape(label)}\]\s*(.*?)(?=\s*\[(?:{headers})\]|$)"
        match = re.search(pattern, text, flags=re.DOTALL)

        if match:
            value = clean_extracted_value(match.group(1))
            if value:
                return value

    return ""


# ------------------------------------------------------------
# 본문 안의 colon 필드 추출
# ------------------------------------------------------------
# 예:
# 정책명: 청년월세 지원사업
# 지원내용: 월세 일부 지원
# ------------------------------------------------------------

def extract_colon_field(text: str, labels: List[str]) -> str:
    """
    정책명: ..., 지원내용: ... 같은 colon 필드를 추출한다.
    """
    if not text:
        return ""

    for label in labels:
        # 한 줄형 필드 우선
        pattern_line = rf"{re.escape(label)}\s*[:：]\s*([^\n\r]+)"
        match = re.search(pattern_line, text)

        if match:
            value = clean_extracted_value(match.group(1))
            if value:
                return value

        # 여러 필드가 같은 줄에 붙어 있는 경우 보완
        pattern_until_field = (
            rf"{re.escape(label)}\s*[:：]\s*(.*?)"
            rf"(?=\s+(?:정책명|서비스명|카테고리|내용|상세 URL|URL|지원대상|지원내용|신청 방법|신청방법|선정기준)\s*[:：]|$)"
        )
        match = re.search(pattern_until_field, text, flags=re.DOTALL)

        if match:
            value = clean_extracted_value(match.group(1))
            if value:
                return value

    return ""


def extract_field_from_text(text: str, labels: List[str]) -> str:
    """
    colon 형식과 bracket 형식을 모두 지원한다.
    """
    return extract_colon_field(text, labels) or extract_bracket_field(text, labels)


# ------------------------------------------------------------
# 정책명 추출
# ------------------------------------------------------------

def extract_policy_name(payload: Dict[str, Any], raw_text: str) -> Tuple[str, str, int]:
    """
    정책명을 추출한다.

    반환값:
    - policy_name: 추출된 정책명
    - source: 어디서 추출했는지
    - priority: 낮을수록 신뢰도 높음
    """
    payload_name = get_payload_value(
        payload,
        "policy_name",
        "title",
        "name",
        "service_name",
        "servNm",
        "정책명",
        "서비스명",
        "제목",
    )

    if payload_name:
        return clean_extracted_value(payload_name), "payload", 0

    colon_policy = extract_colon_field(raw_text, ["정책명", "제목"])
    if colon_policy:
        return colon_policy, "policy_colon", 1

    bracket_policy = extract_bracket_field(raw_text, ["정책명", "제목"])
    if bracket_policy:
        return bracket_policy, "policy_bracket", 1

    colon_service = extract_colon_field(raw_text, ["서비스명"])
    if colon_service:
        return colon_service, "service_colon", 2

    bracket_service = extract_bracket_field(raw_text, ["서비스명"])
    if bracket_service:
        return bracket_service, "service_bracket", 2

    return "정책명 확인 필요", "missing", 9


# ------------------------------------------------------------
# 정책 핵심 내용 추출
# ------------------------------------------------------------

def extract_policy_content(payload: Dict[str, Any], raw_text: str) -> Tuple[str, str]:
    """
    모델과 reranker에 사용할 핵심 내용을 추출한다.
    raw_text 전체보다 지원대상/지원내용/신청정보 중심으로 정리하는 것이 목표다.
    """
    # 1순위: 명시적 내용 필드
    content = extract_colon_field(raw_text, ["내용"])
    if content:
        return content, "content_colon"

    # 2순위: [지원내용]
    content = extract_bracket_field(raw_text, ["지원내용"])
    if content:
        return content, "support_content_bracket"

    # 3순위: 지원내용:
    content = extract_colon_field(raw_text, ["지원내용", "지원 내용"])
    if content:
        return content, "support_content_colon"

    # 4순위: payload summary/content 계열
    payload_content = get_payload_value(
        payload,
        "summary",
        "servDgst",
        "description",
        "desc",
        "content",
        "내용",
        "지원내용",
    )

    if payload_content and payload_content != raw_text:
        return normalize_policy_text(payload_content), "payload_content"

    # 5순위: 지원대상 + 지원내용 + 신청정보 조합
    parts = []

    target = extract_field_from_text(raw_text, ["지원대상", "지원 대상"])
    support = extract_field_from_text(raw_text, ["지원내용", "지원 내용"])
    apply_info = extract_field_from_text(raw_text, ["신청정보", "신청방법", "신청 방법"])

    if target:
        parts.append(f"지원대상: {target}")

    if support:
        parts.append(f"지원내용: {support}")

    if apply_info:
        parts.append(f"신청정보: {apply_info}")

    if parts:
        return " / ".join(parts), "combined_fields"

    # 6순위: 어쩔 수 없을 때 raw text 사용
    return normalize_policy_text(raw_text), "raw_text"


# ------------------------------------------------------------
# URL / 지역 / 섹션 추출
# ------------------------------------------------------------

def extract_url_from_text(text: str) -> str:
    """
    payload에 URL이 없을 때 본문에서 URL을 찾는다.
    """
    if not text:
        return ""

    explicit = extract_field_from_text(text, ["상세 URL", "신청 링크", "URL", "url"])
    if explicit:
        return explicit

    match = re.search(r"https?://[^\s\]]+", text)
    if match:
        return match.group(0).strip()

    return ""


def extract_region(payload: Dict[str, Any], raw_text: str) -> str:
    """
    지역 정보를 추출한다.
    """
    region = get_payload_value(
        payload,
        "region",
        "area",
        "지역",
        "지원 범위",
        "필터링",
    )

    if not region:
        region = " ".join(filter(None, [
            get_payload_value(payload, "region_sido"),
            get_payload_value(payload, "region_sigungu"),
        ]))
    if not region:
        region = extract_field_from_text(raw_text, ["지역", "지원 범위", "필터링"])
    if not region and get_payload_value(payload, "scope") in {"national", "nationwide", "전국"}:
        region = "전국"

    region = clean_extracted_value(region)
    region = re.sub(r"^지역\s+", "", region).strip()
    return region


def extract_section(payload: Dict[str, Any], raw_text: str) -> str:
    """
    정책 분류/섹션 정보를 추출한다.
    """
    section = get_payload_value(
        payload,
        "section",
        "category",
        "카테고리",
        "분류",
        "분야",
        "질문 유형",
        "설명 구분",
        "섹션",
    )

    if not section:
        section = extract_field_from_text(
            raw_text,
            ["카테고리", "분류", "분야", "질문 유형", "설명 구분", "섹션"]
        )

    return clean_extracted_value(section)


# ------------------------------------------------------------
# 검색 결과 1개를 표준 dict로 변환
# ------------------------------------------------------------

def normalize_qdrant_hit(hit: Any, collection_name: str) -> Dict[str, Any]:
    """
    Qdrant 검색 결과 hit 하나를 우리 RAG 파이프라인에서 사용할 표준 형태로 변환한다.
    """
    payload = getattr(hit, "payload", None) or {}
    score = getattr(hit, "score", 0.0)
    point_id = getattr(hit, "id", "")

    raw_text = get_payload_value(
        payload,
        "embedding_text",
        "text",
        "chunk_text",
        "content",
        "page_content",
        "reranker_text",
    )
    # 필드 경계를 추출하기 전에는 원본 줄바꿈을 보존한다.
    raw_text = raw_text.strip()

    policy_name, name_source, name_priority = extract_policy_name(payload, raw_text)
    is_policy_unit = bool(get_payload_value(payload, "embedding_text", "reranker_text"))
    content = raw_text
    if is_policy_unit:
        # 정책 단위 본문은 지원내용만 떼지 않고 자격·예외·신청 정보를 보존한다.
        content_source = "policy_unit_full_text"
    else:
        content_source = "legacy_full_text"

    url = get_payload_value(
        payload,
        "url",
        "source_url",
        "link",
        "servDtlLink",
        "상세 URL",
        "신청 링크",
    )

    if not url:
        url = extract_url_from_text(raw_text)

    region = extract_region(payload, raw_text)
    section = extract_section(payload, raw_text)

    content = normalize_policy_text(content)

    # 내용이 없으면 이후 단계에서 제외할 수 있도록 빈 dict 반환
    if not content:
        return {}

    # 정책명이 잘 잡힌 문서를 약간 우대하기 위한 보정값
    if name_priority <= 1:
        priority_bonus = 0.02
    elif name_priority == 2:
        priority_bonus = 0.0
    else:
        priority_bonus = -0.03

    score_value = float(score) if score is not None else 0.0

    return {
        "point_id": str(point_id),
        "service_id": get_payload_value(payload, "service_id"),
        "schema_version": get_payload_value(payload, "schema_version"),
        "scope": get_payload_value(payload, "scope"),
        "region_sido": get_payload_value(payload, "region_sido"),
        "region_sigungu": get_payload_value(payload, "region_sigungu"),
        "content_hash": get_payload_value(payload, "content_hash"),
        "preserve_full_text": is_policy_unit,
        "reranker_text": get_payload_value(payload, "reranker_text"),
        "collection": collection_name,
        "score": score_value,
        "adjusted_score": score_value + priority_bonus,
        "policy_name": policy_name,
        "name_source": name_source,
        "name_priority": name_priority,
        "content_source": content_source,
        "text": content,
        "raw_text": raw_text,
        "url": url,
        "region": region,
        "section": section,
    }


# ------------------------------------------------------------
#  Reranker 입력 문서 구성
# ------------------------------------------------------------

def build_reranker_document_text(item: Dict[str, Any]) -> str:
    """
    Reranker가 질문과 비교할 문서 텍스트를 구성한다.

    Reranker는 질문-문서 pair를 보고 관련성을 판단하므로,
    정책명/지역/구분/핵심 내용을 함께 넣어주는 것이 좋다.
    """
    fields = (
        ("정책명", "policy_name"),
        ("지역", "region"),
        ("구분", "section"),
        ("내용", "text"),
    )
    parts = [f"{label}: {item[key]}" for label, key in fields if item.get(key)]
    return "\n".join(parts).strip()


print("Payload 정제 함수 준비 완료")


Payload 정제 함수 준비 완료


In [13]:
# ============================================================
# [셀 14/26] Qdrant 1차 후보 검색 함수
# ============================================================
# 이 셀은 사용자 질문을 embedding한 뒤,
# Qdrant에서 관련 정책 문서 후보를 넉넉히 가져온다.
#
# [RAG 설계상 역할]
# - 이 단계는 2-stage RAG 중 1단계 dense retrieval이다.
# - 목표는 "정답 문서를 놓치지 않도록" 후보를 넓게 가져오는 것이다.
# - 최종 선택은 다음 단계의 Reranker가 담당한다.
#
# - BAAI/bge-m3: 사용자 질문을 dense vector로 변환
# - Qdrant: 정책 문서 vector search 수행
#
# - 중앙정부/지자체 컬렉션을 모두 검색한다.
# - 각 Qdrant hit를 셀 13의 normalize_qdrant_hit()으로 표준 형태로 정리한다.
# - 이후 reranker가 사용할 후보 목록을 만든다.
# ============================================================

from typing import Optional

import time

try:
    import torch
except Exception:
    torch = None


def _sync_cuda_for_metrics() -> None:
    if torch is not None and torch.cuda.is_available():
        torch.cuda.synchronize()


def _empty_retrieval_metrics() -> dict:
    return {
        "embedding_ms": 0.0,
        "qdrant_query_ms": 0.0,
        "candidate_postprocess_ms": 0.0,
        "qdrant_query_by_collection_ms": {},
        "qdrant_raw_hits_by_collection": {},
        "retrieval_candidate_count": 0,
        "qdrant_fetch_k": globals().get("QDRANT_FETCH_K", None),
        "qdrant_vector_name": globals().get("QDRANT_VECTOR_NAME", None),
    }


# ------------------------------------------------------------
# Qdrant query 래퍼
# ------------------------------------------------------------
# 클라이언트에 query_points가 있으면 그 메서드를 쓰고, 없으면 search를 쓴다.
# ------------------------------------------------------------

def qdrant_dense_search(
    collection_name: str,
    query_vector: list,
    limit: int,
) -> list:
    """
    Qdrant 컬렉션 하나에서 dense vector search를 수행한다.

    반환:
    - Qdrant hit 목록
    """
    # query_points()를 지원하는 qdrant-client 버전
    if hasattr(qdrant_client, "query_points"):
        kwargs = {
            "collection_name": collection_name,
            "query": query_vector,
            "limit": limit,
            "with_payload": True,
        }

        # named vector 컬렉션을 사용하는 경우
        if QDRANT_VECTOR_NAME:
            kwargs["using"] = QDRANT_VECTOR_NAME

        result = qdrant_client.query_points(**kwargs)

        # qdrant-client 버전에 따라 points 속성에 결과가 들어감
        return getattr(result, "points", result)

    # search() 경로
    kwargs = {
        "collection_name": collection_name,
        "query_vector": query_vector,
        "limit": limit,
        "with_payload": True,
    }

    if QDRANT_VECTOR_NAME:
        kwargs["query_vector"] = (QDRANT_VECTOR_NAME, query_vector)

    return qdrant_client.search(**kwargs)


# ------------------------------------------------------------
# 질문 embedding 생성
# ------------------------------------------------------------

def encode_query(query: str) -> list:
    """
    사용자 질문을 Qdrant 검색용 dense vector로 변환한다.
    """
    query_vector = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0].astype("float32").tolist()

    return query_vector


# ------------------------------------------------------------
# 1차 후보 검색
# ------------------------------------------------------------

def retrieve_candidates(query: str, fetch_k: Optional[int] = None, return_metrics: bool = False):
    retrieval_start = time.perf_counter()
    metrics = _empty_retrieval_metrics()

    if fetch_k is None:
        fetch_k = QDRANT_FETCH_K
    metrics["qdrant_fetch_k"] = fetch_k

    query = (query or "").strip()

    def _finish(candidates: list):
        metrics["retrieval_candidate_count"] = len(candidates)
        metrics["retrieval_ms"] = round((time.perf_counter() - retrieval_start) * 1000, 2)
        return (candidates, metrics) if return_metrics else candidates

    if not query:
        return _finish([])

    _sync_cuda_for_metrics()
    t_embedding = time.perf_counter()
    query_vector = encode_query(query)
    _sync_cuda_for_metrics()
    metrics["embedding_ms"] = round((time.perf_counter() - t_embedding) * 1000, 2)

    candidates = []

    print("\n========== Qdrant retrieval ==========")
    print("query:", query)
    print("fetch_k per collection:", fetch_k)
    print("collections:", QDRANT_COLLECTIONS)

    for collection_name in QDRANT_COLLECTIONS:
        try:
            t_qdrant = time.perf_counter()
            hits = qdrant_dense_search(
                collection_name=collection_name,
                query_vector=query_vector,
                limit=fetch_k,
            )
            qdrant_ms = round((time.perf_counter() - t_qdrant) * 1000, 2)
            metrics["qdrant_query_by_collection_ms"][collection_name] = qdrant_ms
            metrics["qdrant_query_ms"] = round(metrics["qdrant_query_ms"] + qdrant_ms, 2)
            metrics["qdrant_raw_hits_by_collection"][collection_name] = len(hits)

            print(f"\n[COLLECTION] {collection_name}")
            print(f"raw hits: {len(hits)}")
            print(f"query_ms: {qdrant_ms:.2f}")

            t_post = time.perf_counter()
            accepted_count = 0
            for hit in hits:
                item = normalize_qdrant_hit(hit, collection_name)
                if not item:
                    continue
                candidates.append(item)
                accepted_count += 1
            if hits and not accepted_count:
                raise ValueError(
                    f"{collection_name}: 검색 {len(hits)}개가 본문 정제에서 모두 제외됐습니다. "
                    "payload의 embedding_text/text/reranker_text 및 스키마를 확인하세요."
                )
            if len(hits) > accepted_count:
                print(f"[WARN] {collection_name}: 본문 없는 후보 {len(hits) - accepted_count}개 제외")
            metrics["candidate_postprocess_ms"] = round(
                metrics["candidate_postprocess_ms"] + ((time.perf_counter() - t_post) * 1000),
                2,
            )
        except Exception as e:
            print(f"[ERROR] Qdrant search failed: {collection_name}")
            print(f"reason: {e}")
            raise

    print("\nnormalized candidate count:", len(candidates))

    t_post = time.perf_counter()
    candidates.sort(
        key=lambda x: x.get("adjusted_score", x.get("score", 0.0)),
        reverse=True,
    )
    metrics["candidate_postprocess_ms"] = round(
        metrics["candidate_postprocess_ms"] + ((time.perf_counter() - t_post) * 1000),
        2,
    )

    print("\n[retrieval top candidates]")
    for i, item in enumerate(candidates[:10], 1):
        print(
            f"{i}. [{item.get('collection')}] "
            f"{item.get('policy_name')} | "
            f"score={item.get('score'):.4f} | "
            f"adjusted={item.get('adjusted_score'):.4f} | "
            f"region={item.get('region') or '-'}"
        )

    print(
        "retrieval detail: "
        f"embedding={metrics['embedding_ms']:.2f}ms | "
        f"qdrant={metrics['qdrant_query_ms']:.2f}ms | "
        f"postprocess={metrics['candidate_postprocess_ms']:.2f}ms"
    )
    print("==========================================\n")

    return _finish(candidates)


print("Qdrant 1차 후보 검색 함수 준비 완료")


Qdrant 1차 후보 검색 함수 준비 완료


In [14]:
# ============================================================
# [셀 15/26] Reranker 재정렬 함수
# ============================================================
# 이 셀은 Qdrant 1차 검색 후보를 Reranker로 다시 정렬한다.
#
# [RAG 설계상 역할]
# - Qdrant dense retrieval은 빠르게 후보를 넓게 가져오는 단계이다.
# - Reranker는 질문과 후보 문서를 pair로 함께 보고 관련성 점수를 다시 계산한다.
# - 최종 LLM context에는 Reranker 점수가 높은 문서를 우선 사용한다.
#
# [외부 공개 모델 활용]
# - Dongjin-kr/ko-reranker
# - 역할: 질문-문서 pair의 관련성 점수 계산
#
# - Qdrant hit에서 추출한 정책명/지역/구분/내용을 조합해
#   Reranker 입력용 문서 텍스트를 구성한다.
# - Reranker 점수를 각 후보 item에 reranker_score로 저장한다.
# ============================================================

import math
import torch
from typing import List, Dict, Any


# ------------------------------------------------------------
#  Reranker 점수 계산
# ------------------------------------------------------------

def compute_reranker_scores(
    query: str,
    candidates: List[Dict[str, Any]],
    batch_size: int = 8,
) -> List[float]:
    """
    사용자 질문과 후보 문서 목록을 pair로 만들어 reranker 점수를 계산한다.

    입력:
    - query: 사용자 질문
    - candidates: Qdrant 1차 검색 후보 목록
    - batch_size: reranker 추론 batch 크기

    출력:
    - 후보 문서별 reranker 점수 리스트
    """
    if not candidates:
        return []

    query = (query or "").strip()
    scores = []

    reranker_device = next(reranker_model.parameters()).device

    for start in range(0, len(candidates), batch_size):
        batch = candidates[start:start + batch_size]

        # Reranker에 넣을 질문-문서 pair 구성
        query_texts = [query] * len(batch)
        doc_texts = [build_reranker_document_text(item) for item in batch]

        inputs = reranker_tokenizer(
            query_texts,
            doc_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )

        inputs = {
            key: value.to(reranker_device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            outputs = reranker_model(**inputs)

            # 일반적인 reranker는 logits가 [batch, 1] 또는 [batch] 형태로 나온다.
            logits = outputs.logits

            if logits.ndim == 2 and logits.shape[1] == 1:
                batch_scores = logits.squeeze(-1)
            elif logits.ndim == 2 and logits.shape[1] > 1:
                # 혹시 2-class classification 형태라면 positive class 점수를 사용
                batch_scores = logits[:, -1]
            else:
                batch_scores = logits.view(-1)

        scores.extend(batch_scores.detach().float().cpu().tolist())

    return scores


# ------------------------------------------------------------
# 후보 목록 재정렬
# ------------------------------------------------------------

def normalize_reranker_score(raw_score: float) -> float:
    """
    reranker raw logit을 0~1로 정규화한다.

    ko-reranker는 CrossEntropy로 학습되어 출력 범위가 정해져 있지 않다.
    모델 카드는 exp_normalize(후보 전체 softmax)를 권장하지만, 그 값은
    후보 개수와 구성에 따라 같은 문서의 점수도 달라져 고정 기준과 비교할 수 없다.
    여기서는 질문-문서 쌍마다 독립적으로 계산되는 sigmoid를 쓴다.

    오버플로를 피하기 위해 부호에 따라 식을 나눈다.
    """
    if raw_score >= 0:
        return 1.0 / (1.0 + math.exp(-raw_score))
    exp_value = math.exp(raw_score)
    return exp_value / (1.0 + exp_value)


def rerank_candidates(
    query: str,
    candidates: List[Dict[str, Any]],
    top_n: int = None,
) -> List[Dict[str, Any]]:
    """
    Qdrant 1차 후보를 Reranker 점수 기준으로 재정렬한다.

    흐름:
    1. 각 후보 문서에 대해 질문-문서 pair 생성
    2. Dongjin-kr/ko-reranker로 관련성 점수 계산
    3. item["reranker_score"]에 점수 저장
    4. reranker_score 기준 내림차순 정렬
    5. 상위 top_n개 반환
    """
    if top_n is None:
        top_n = RERANK_TOP_N

    if not candidates:
        return []

    print("\n========== Reranker 재정렬 ==========")
    print("query:", query)
    print("input candidates:", len(candidates))
    print("top_n:", top_n)

    scores = compute_reranker_scores(
        query=query,
        candidates=candidates,
        batch_size=8,
    )

    reranked = []

    for item, score in zip(candidates, scores):
        new_item = dict(item)
        # reranker_score는 0~1 정규화 값, reranker_score_raw는 원본 logit이다.
        # sigmoid는 단조 증가 함수이므로 정렬 순서는 바뀌지 않는다.
        raw_score = float(score)
        new_item["reranker_score_raw"] = raw_score
        new_item["reranker_score"] = normalize_reranker_score(raw_score)
        reranked.append(new_item)

    # Reranker 점수 기준 정렬
    reranked.sort(
        key=lambda x: x.get("reranker_score", -math.inf),
        reverse=True,
    )

    # 상위 top_n만 반환
    reranked = reranked[:top_n]

    print("\n[Reranker 상위 후보]")
    for i, item in enumerate(reranked[:10], 1):
        print(
            f"{i}. [{item.get('collection')}] "
            f"{item.get('policy_name')} | "
            f"reranker={item.get('reranker_score'):.4f} "
            f"(raw {item.get('reranker_score_raw', float('nan')):.4f}) | "
            f"qdrant={item.get('score'):.4f} | "
            f"region={item.get('region') or '-'}"
        )

    print("=====================================\n")

    return reranked


print("Reranker 재정렬 함수 준비 완료")


Reranker 재정렬 함수 준비 완료


In [15]:
# ============================================================
# [셀 16/26] 최종 LLM context 구성 함수
# ============================================================
# 이 셀은 Reranker가 재정렬한 후보 문서 중에서
# 최종적으로 LLM에게 제공할 정책 문서 context를 만든다.
#
# [RAG 설계상 역할]
# - 셀 14: Qdrant 1차 후보 검색
# - 셀 15: Dongjin-kr/ko-reranker로 질문 관련성 재정렬
# - 셀 16: 중복 정책 제거 + 최종 문서 선택 + LLM context 구성
#
# - 같은 정책이 여러 chunk로 검색될 수 있으므로 중복을 줄인다.
# - Reranker 점수가 높은 정책을 우선 사용한다.
# - Qwen에게 너무 긴 문서를 넣지 않도록 정책별 길이를 제한한다.
# - 정책명/지역/구분/내용을 정리해 LLM이 읽기 쉬운 context로 만든다.
# ============================================================

from typing import List, Dict, Any


# ------------------------------------------------------------
# 중복 정책 제거
# ------------------------------------------------------------
# Qdrant는 chunk 단위로 검색하므로 같은 정책의 여러 조각이 검색될 가능성 미연에 방지
# 동일 정책이 context를 과도하게 차지하지 않도록 정책명+URL 기준으로 묶는다.
# ------------------------------------------------------------

def group_chunks_by_policy(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    reranker로 정렬된 후보들을 정책 단위로 묶는다.

    반환:
    - 정책별 대표 정보와 관련 chunk 목록을 가진 리스트
    """
    grouped = {}

    for item in items:
        policy_name = item.get("policy_name") or "정책명 확인 필요"
        url = item.get("url") or ""

        # 정책명과 URL이 같으면 같은 정책으로 간주
        # URL이 비어 있더라도 정책명이 같으면 우선 같은 정책으로 묶인다.
        group_key = (policy_name, url)

        if group_key not in grouped:
            grouped[group_key] = {
                "policy_name": policy_name,
                "url": url,
                "collection": item.get("collection", ""),
                "region": item.get("region", ""),
                "section": item.get("section", ""),
                "best_qdrant_score": item.get("score", 0.0),
                "best_reranker_score": item.get("reranker_score", 0.0),
                "best_reranker_score_raw": item.get("reranker_score_raw", 0.0),
                "chunks": [],
            }

        grouped_item = grouped[group_key]

        # 같은 정책 안에서는 가장 높은 점수를 보존
        grouped_item["best_qdrant_score"] = max(
            grouped_item.get("best_qdrant_score", 0.0),
            item.get("score", 0.0),
        )

        # 0~1 점수 중 최댓값을 고르고, 그 chunk의 raw logit을 함께 저장한다.
        chunk_rerank_score = item.get("reranker_score", 0.0)
        if chunk_rerank_score >= grouped_item.get("best_reranker_score", 0.0):
            grouped_item["best_reranker_score"] = chunk_rerank_score
            grouped_item["best_reranker_score_raw"] = item.get("reranker_score_raw", 0.0)

        # region/section이 비어 있으면 후속 chunk에서 보완
        for field in ("region", "section"):
            if not grouped_item.get(field) and item.get(field):
                grouped_item[field] = item[field]

        grouped_item["chunks"].append(item)

    # Reranker 점수가 높은 정책 순으로 정렬
    return sorted(
        grouped.values(),
        key=lambda item: item.get("best_reranker_score", 0.0),
        reverse=True,
    )


# ------------------------------------------------------------
# 정책별 context 텍스트 구성
# ------------------------------------------------------------
# LLM에게 그대로 raw text를 넣지 않고,
# 정책명/지역/구분/내용 중심으로 정리해서 넣는다.
# ------------------------------------------------------------

def build_policy_context_text(policy_item: Dict[str, Any]) -> str:
    """
    정책 하나에 대해 Qwen에게 넣을 context 텍스트를 구성한다.
    """
    policy_name = policy_item.get("policy_name", "정책명 확인 필요")
    region = policy_item.get("region", "")
    section = policy_item.get("section", "")
    chunks = policy_item.get("chunks", [])

    # 같은 정책의 여러 chunk 내용을 이어붙임
    chunk_texts = []
    seen_texts = set()

    for chunk in chunks:
        text = (chunk.get("text") or "").strip()

        if not text:
            continue

        # 완전히 같은 내용 또는 거의 같은 내용 중복 방지
        normalized = text[:200]

        if normalized in seen_texts:
            continue

        seen_texts.add(normalized)
        chunk_texts.append(text)

    joined_text = " ".join(chunk_texts).strip()

    # 정책별 context 길이 제한
    # 너무 긴 정책 문서가 전체 prompt를 과도하게 차지하지 않도록 제한한다.
    if len(joined_text) > MAX_CONTEXT_CHARS_PER_POLICY:
        joined_text = joined_text[:MAX_CONTEXT_CHARS_PER_POLICY].rstrip() + "..."

    parts = [
        f"정책명: {policy_name}",
    ]

    if region:
        parts.append(f"지역: {region}")

    if section:
        parts.append(f"구분: {section}")

    if joined_text:
        parts.append(f"내용: {joined_text}")

    return "\n".join(parts).strip()


# ------------------------------------------------------------
# 최종 LLM context 구성
# ------------------------------------------------------------
# Reranker가 재정렬한 후보 중 최종 top-k 정책만 골라
# Qwen에게 제공할 [문서] context를 만든다.
# ------------------------------------------------------------

def build_final_context(
    reranked_items: List[Dict[str, Any]],
    final_top_k: int = None,
) -> Dict[str, Any]:
    """
    Reranker가 정렬한 후보 문서를 최종 LLM context로 변환한다.

    반환:
    {
        "context_text": LLM에 넣을 문서 문자열,
        "selected_policies": 최종 선택된 정책 목록
    }
    """
    if final_top_k is None:
        final_top_k = FINAL_TOP_K

    if not reranked_items:
        return {
            "context_text": "검색된 관련 정책 문서가 없습니다.",
            "selected_policies": [],
        }

    # 같은 정책의 여러 chunk를 하나로 묶음
    grouped_policies = group_chunks_by_policy(reranked_items)

    # 최종 top-k 정책만 사용
    selected_policies = grouped_policies[:final_top_k]

    context_blocks = []

    for idx, policy in enumerate(selected_policies, 1):
        policy_text = build_policy_context_text(policy)

        if policy_text:
            context_blocks.append(
                f"[문서 {idx}]\n{policy_text}"
            )

    context_text = "\n\n".join(context_blocks).strip()

    if not context_text:
        context_text = "검색된 관련 정책 문서가 없습니다."

    # 디버깅용 로그
    # 답변 화면에는 참고문서 블록을 붙이지 않지만,
    # Colab 로그에서는 어떤 정책이 최종 선택되었는지 확인할 수 있게 한다.
    print("\n========== 최종 LLM context 구성 ==========")
    print("최종 선택 정책 수:", len(selected_policies))

    for i, policy in enumerate(selected_policies, 1):
        print(
            f"{i}. {policy.get('policy_name')} | "
            f"reranker={policy.get('best_reranker_score'):.4f} "
            f"(raw {policy.get('best_reranker_score_raw', float('nan')):.4f}) | "
            f"region={policy.get('region') or '-'} | "
            f"collection={policy.get('collection') or '-'}"
        )

    print("==========================================\n")

    return {
        "context_text": context_text,
        "selected_policies": selected_policies,
    }


print("최종 context 구성 함수 준비 완료")


최종 context 구성 함수 준비 완료


In [16]:
# ============================================================
# [셀 17/26] RAG 기반 답변 생성 함수
# ============================================================
# 이 셀은 RAG 답변 함수와 grounded system prompt를 정의한다.
# generate_with_qwen은 셀 20에서 vLLM 호출로 정의한다.
#
# [전체 RAG 흐름]
# 1. 사용자 질문 입력
# 2. Qdrant dense retrieval로 후보 문서 검색
# 3. Dongjin-kr/ko-reranker로 후보 문서 재정렬
# 4. 최종 LLM context 구성
# 5. LLM이 답변 생성
#
# [외부 공개 모델 활용]
# - Qwen: 최종 답변 생성
# - BAAI/bge-m3: Qdrant 검색용 질문 embedding
# - Dongjin-kr/ko-reranker: 후보 문서 재정렬
#
# - 정책 상담용 system prompt 작성
# - 검색 문서 context 구성
# - 문서 기반 답변 원칙 적용
# - 참고문서 블록은 답변에 붙이지 않음
# ============================================================


# ------------------------------------------------------------
#정책 상담용 system prompt
# ------------------------------------------------------------
# 모델이 검색 문서를 바탕으로 자연스럽게 답변하도록 제어한다.
# ------------------------------------------------------------

SYSTEM_PROMPT = """
너는 대한민국 청년 정책을 안내하는 친절하고 정확한 정책 상담사야.

반드시 제공된 [문서] 내용만을 근거로 답변해.
문서에 없는 내용은 추측하지 말고, 정보가 부족하다고 말해.
DB 필드명을 그대로 나열하지 말고, 사람이 이해하기 쉬운 자연스러운 문장으로 답변해.
사용자의 상황과 가장 관련 있는 정책을 중심으로 설명해.
반드시 한국어로만 답변해.

답변은 너무 짧게 끝내지 말고, 사용자가 실제로 이해하고 다음 행동을 정할 수 있도록 충분히 설명해.
다만 문서에 없는 신청 자격, 금액, 기간, 링크, 절차를 임의로 만들어내지 마.

답변 구조는 다음을 따르는 것을 권장해.
사용자의 상황에 맞는 정책이 있는지 먼저 요약
관련 정책이 있다면 정책명과 지원 내용을 설명
지원 대상이나 조건이 문서에 있으면 함께 설명
신청 방법이나 확인해야 할 사항이 문서에 있으면 안내
문서에 부족한 정보가 있으면 추가 확인이 필요하다고 말하기

답변은 보통 2~4개 단락으로 작성해.
관련 정책이 여러 개라면 정책별로 나누어 설명해.
""".strip()


# ------------------------------------------------------------
# prompt 구성
# ------------------------------------------------------------

def build_messages_for_generation(query: str, context_text: str) -> list:
    """
    LLM에 전달할 chat message 목록을 구성한다.
    """
    user_content = f"""
[문서]
{context_text}

[사용자 질문]
{query}
""".strip()

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_content,
        },
    ]


# ------------------------------------------------------------
# LLM 답변 생성
# ------------------------------------------------------------

def generate_with_qwen(query: str, context_text: str) -> tuple:
    raise RuntimeError("셀 18~20(vLLM 병합·서버·함수 정의)을 먼저 실행하세요.")


# ------------------------------------------------------------
# Frontend retrieval/reranker 디버그 payload
# ------------------------------------------------------------

# ------------------------------------------------------------

def make_retrieval_debug_item(item: dict, rank: int) -> dict:
    """
    Qdrant/Reranker 후보 문서를 프론트 표시용으로 정리한다.
    """
    text = (item.get("text") or "").strip()

    if len(text) > 220:
        content_preview = text[:220].rstrip() + "..."
    else:
        content_preview = text

    return {
        "rank": rank,
        "collection": item.get("collection", ""),
        "policy_name": item.get("policy_name", "정책명 확인 필요"),
        "region": item.get("region", ""),
        "section": item.get("section", ""),
        "qdrant_score": float(item.get("score", 0.0) or 0.0),
        "adjusted_score": float(item.get("adjusted_score", item.get("score", 0.0)) or 0.0),
        "reranker_score": (
            float(item.get("reranker_score"))
            if item.get("reranker_score") is not None
            else None
        ),
        "content_preview": content_preview,
    }


def build_retrieval_debug(candidates: list, reranked_items: list, limit: int = 5) -> dict:
    """
    Rerank 전후 Top N 후보를 프론트로 보내기 위한 형태로 만든다.
    """
    return {
        "before_rerank": [
            make_retrieval_debug_item(item, rank=rank)
            for rank, item in enumerate((candidates or [])[:limit], 1)
        ],
        "after_rerank": [
            make_retrieval_debug_item(item, rank=rank)
            for rank, item in enumerate((reranked_items or [])[:limit], 1)
        ],
    }

# ------------------------------------------------------------
# 전체 RAG 답변 생성 함수
# ------------------------------------------------------------
import time
rag_cache = {}
#captum 계산을 위한 cache 저장


def _empty_ask_metrics() -> dict:
    return {
        "embedding_ms": 0.0,
        "qdrant_query_ms": 0.0,
        "candidate_postprocess_ms": 0.0,
        "qdrant_query_by_collection_ms": {},
        "qdrant_raw_hits_by_collection": {},
        "retrieval_candidate_count": 0,
        "retrieval_ms": 0.0,
        "rerank_ms": 0.0,
        "prompt_build_ms": 0.0,
        "generation_ms": 0.0,
        "generated_tokens": 0,
        "generation_tps": 0.0,
        "ask_total_ms": 0.0,
        "cache_hit": False,
        "hybrid_attention": bool(globals().get("USE_HYBRID_ATTENTION", False)),
        "answer_attention_backend": globals().get("ANSWER_ATTENTION_DETAIL", "unknown"),
        "xai_attention_backend": globals().get("XAI_ATTENTION_DETAIL", "unknown"),
        "answer_lora_status": globals().get("ANSWER_LORA_STATUS", "unknown"),
        "xai_lora_status": globals().get("XAI_LORA_STATUS", "unknown"),
    }


def _print_pipeline_metrics(metrics: dict) -> None:
    print("\n========== pipeline metrics ==========")
    print(f"embedding_ms   : {metrics.get('embedding_ms', 0):.2f} ms")
    print(f"qdrant_query_ms: {metrics.get('qdrant_query_ms', 0):.2f} ms")
    print(f"postprocess_ms : {metrics.get('candidate_postprocess_ms', 0):.2f} ms")
    print(f"retrieval_ms   : {metrics.get('retrieval_ms', 0):.2f} ms")
    print(f"rerank_ms      : {metrics.get('rerank_ms', 0):.2f} ms")
    print(f"prompt_build_ms: {metrics.get('prompt_build_ms', 0):.2f} ms")
    print(f"generation_ms  : {metrics.get('generation_ms', 0):.2f} ms")
    print(f"generated_tokens: {metrics.get('generated_tokens', 0)}")
    print(f"generation_tps : {metrics.get('generation_tps', 0):.2f} tokens/sec")
    print(f"ask_total_ms   : {metrics.get('ask_total_ms', 0):.2f} ms")
    print(f"qdrant_by_col  : {metrics.get('qdrant_query_by_collection_ms', {})}")
    print(f"answer_backend : {metrics.get('answer_attention_backend', 'unknown')}")
    print(f"xai_backend    : {metrics.get('xai_attention_backend', 'unknown')}")
    print(f"answer_lora    : {metrics.get('answer_lora_status', 'unknown')}")
    print(f"xai_lora       : {metrics.get('xai_lora_status', 'unknown')}")
    print("====================================\n")


def generate_rag_answer(query: str) -> dict:
    """
    사용자 질문에 대해 RAG 기반 정책 상담 답변을 생성한다.

    흐름:
    1. Qdrant 1차 후보 검색
    2. Dongjin-kr/ko-reranker 재정렬
    3. 최종 LLM context 구성
    4. LLM 답변 생성
    """
    query = (query or "").strip()

    if not query:
        return {
            "answer": "질문 내용이 비어 있습니다. 궁금한 정책이나 상황을 입력해 주세요.",
            "retrieval_debug": build_retrieval_debug([], []),
            "metrics": _empty_ask_metrics(),
        }

    print("\n\n==================================================")
    print("[RAG 답변 생성 시작]")
    print("질문:", query)
    print("==================================================")

    ask_start = time.perf_counter()
    metrics = _empty_ask_metrics()

    # 1. Qdrant retrieval
    t0 = time.perf_counter()
    retrieval_result = retrieve_candidates(query, return_metrics=True)
    if isinstance(retrieval_result, tuple):
        candidates, retrieval_metrics = retrieval_result
        metrics.update(retrieval_metrics)
    else:
        candidates = retrieval_result
        metrics["retrieval_ms"] = round((time.perf_counter() - t0) * 1000, 2)

    if not candidates:
        metrics["ask_total_ms"] = round((time.perf_counter() - ask_start) * 1000, 2)
        print("[WARN] Qdrant 검색 결과가 없습니다.")
        return {
            "answer": (
                "현재 질문과 직접 관련된 정책 문서를 찾지 못했습니다. "
                "지역, 나이, 현재 상태를 조금 더 구체적으로 적어주시면 다시 확인해볼 수 있습니다."
            ),
            "retrieval_debug": build_retrieval_debug([], []),
            "metrics": metrics,
        }

    # 2. ko-reranker 재정렬
    t0 = time.perf_counter()
    reranked_items = rerank_candidates(query, candidates)
    metrics["rerank_ms"] = round((time.perf_counter() - t0) * 1000, 2)

    if not reranked_items:
        print("[WARN] Reranker 재정렬 결과가 없습니다.")
        return {
            "answer": (
                "검색된 정책 문서를 재정렬하는 과정에서 관련 문서를 확정하지 못했습니다. "
                "질문을 조금 더 구체적으로 입력해 주세요."
            ),
            "retrieval_debug": build_retrieval_debug(candidates, []),
            "metrics": metrics,
        }

    # 3. 최종 context 구성
    t0 = time.perf_counter()
    context_result = build_final_context(reranked_items)
    metrics["prompt_build_ms"] = round((time.perf_counter() - t0) * 1000, 2)
    context_text = context_result["context_text"]
    selected_policies = context_result["selected_policies"]

    if not selected_policies:
        print("[WARN] 최종 선택 정책이 없습니다.")
        return {
            "answer": (
                "질문과 관련된 정책 문서를 충분히 찾지 못했습니다. "
                "지역, 나이, 취업 여부 등 조건을 함께 입력해 주세요."
            ),
            "retrieval_debug": build_retrieval_debug(candidates, reranked_items, limit=5),
            "metrics": metrics,
        }

    # 4. LLM 답변 생성
    answer, gen_metrics = generate_with_qwen(query, context_text)
    metrics.update(gen_metrics)

    if not answer:
        answer = (
            "답변을 생성하지 못했습니다. "
            "질문을 조금 더 구체적으로 입력해 주세요."
        )

    # 참고문서 블록은 붙이지 않는다.
    # 단, 최종 선택 정책은 Colab 로그에서 확인 가능하다.
    metrics["ask_total_ms"] = round((time.perf_counter() - ask_start) * 1000, 2)
    _print_pipeline_metrics(metrics)

    print("[RAG 답변 생성 완료]")
    print("선택 정책 수:", len(selected_policies))
    print("답변:", answer[:500])
    print("==================================================\n")

    # --- 캐시 저장 (XAI 분석 최적화용) ---
    # analyze_with_captum 호출 시 동일한 문맥과 답변을 보장하고 재연산을 방지합니다.
    rag_cache[query] = {
        "context_text": context_text,
        "selected_policies": selected_policies,
        "answer": answer,
        "timestamp": time.time()
    }
    # -------------------------------------

    return {
        "answer": answer,
        "retrieval_debug": build_retrieval_debug(candidates, reranked_items, limit=5),
        "metrics": metrics,
    }


print("RAG 답변 생성 함수 준비 완료")


RAG 답변 생성 함수 준비 완료


In [17]:
# ============================================================
# [셀 18/26] vLLM — LoRA 병합 체크포인트 저장
# - load_qwen_model_for_role은 매 실행마다 merge_and_unload()로 병합하지만
#   결과를 디스크에 저장하지는 않는다.
# - vLLM에 넘길 고정된 병합 체크포인트를 한 번만 만들어 저장한다.
# ============================================================

if "ADAPTER_PATH" not in globals() or "llm_source" not in globals():
    raise RuntimeError("셀 08 실행 필요")

MERGED_MODEL_PATH = LOCAL_MODEL_DIR / "qwen3-4b-instruct-2507-merged"

if MERGED_MODEL_PATH.exists() and any(MERGED_MODEL_PATH.iterdir()):
    print(f"[SKIP] 이미 병합 체크포인트 존재: {MERGED_MODEL_PATH}")
else:
    print("[INFO] LoRA 병합 체크포인트 생성 중...")
    _merge_base = AutoModelForCausalLM.from_pretrained(
        llm_source,
        torch_dtype=TORCH_DTYPE,
        trust_remote_code=True,
        local_files_only=local_files_only,
    )
    if ADAPTER_PATH.exists() and (ADAPTER_PATH / "adapter_config.json").exists():
        _merged = PeftModel.from_pretrained(_merge_base, str(ADAPTER_PATH)).merge_and_unload()
    else:
        print("[WARN] LoRA 어댑터를 찾지 못해 베이스 모델만 저장합니다.")
        _merged = _merge_base

    MERGED_MODEL_PATH.mkdir(parents=True, exist_ok=True)
    _merged.save_pretrained(str(MERGED_MODEL_PATH))
    tokenizer.save_pretrained(str(MERGED_MODEL_PATH))
    del _merge_base, _merged
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"[OK] 병합 체크포인트 저장 완료: {MERGED_MODEL_PATH}")


[SKIP] 이미 병합 체크포인트 존재: /content/drive/MyDrive/Shared_Project/local_models/qwen3-4b-instruct-2507-merged


In [18]:
# ============================================================
# [셀 19/26] vLLM — 격리 가상환경과 서버 기동 (포트 8001)
# - vllm==0.19.*는 torch를 다운그레이드하므로 virtualenv 안에만 설치한다.
# - python -m venv는 이 Colab에서 ensurepip가 실패하므로 virtualenv를 쓴다.
# - 노트북 프로세스 안에서 LLM()을 직접 띄우면 stdout fileno 오류로 죽는다.
#   서버는 `vllm serve` 서브프로세스로만 띄운다.
# - /ask는 vLLM, /analyze는 HF SDPA Feature Ablation이다.
# - 실행용 venv는 /content/vllm_env이다. Drive에는 pip 캐시와 tar만 둔다.
# ============================================================

import json
import os
import shutil
import subprocess
import sys
import urllib.error
import urllib.request
from pathlib import Path

import socket
import time


def _is_port_open(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.5):
            return True
    except OSError:
        return False


def _wait_port(port: int, timeout: float = 15.0) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        if _is_port_open(port):
            return True
        time.sleep(0.3)
    return False

if "MERGED_MODEL_PATH" not in globals():
    raise RuntimeError("셀 18 실행 필요")

if "PROJECT_PATH" not in globals():
    raise RuntimeError("셀 03 실행 필요")

VLLM_VERSION_SPEC = "vllm==0.19.*"
VLLM_VENV = Path("/content/vllm_env")
VLLM_PORT = int(globals().get("VLLM_PORT", 8001))
_python_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
VLLM_CACHE_DIR = PROJECT_PATH / "cache" / "vllm"
PIP_CACHE_DIR = Path(os.environ.get("PIP_CACHE_DIR") or (PROJECT_PATH / "cache" / "pip"))
VLLM_ARCHIVE = VLLM_CACHE_DIR / f"venv_{_python_tag}_vllm019.tar.gz"
VLLM_ARCHIVE_META = VLLM_CACHE_DIR / f"venv_{_python_tag}_vllm019.json"
PIP_CACHE_DIR.mkdir(parents=True, exist_ok=True)
VLLM_CACHE_DIR.mkdir(parents=True, exist_ok=True)

_pip_env = os.environ.copy()
_pip_env["PIP_CACHE_DIR"] = str(PIP_CACHE_DIR)
_pip_env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"


def _vllm_venv_ok(path: Path) -> bool:
    python_bin = path / "bin" / "python"
    vllm_bin = path / "bin" / "vllm"
    return python_bin.is_file() and vllm_bin.is_file()


def _vllm_archive_usable() -> bool:
    if not VLLM_ARCHIVE.is_file() or VLLM_ARCHIVE.stat().st_size < 1024 * 1024:
        return False
    if not VLLM_ARCHIVE_META.is_file():
        return True
    try:
        meta = json.loads(VLLM_ARCHIVE_META.read_text(encoding="utf-8"))
    except Exception:
        return False
    return meta.get("python_tag") == _python_tag and meta.get("vllm_spec") == VLLM_VERSION_SPEC


def _save_vllm_archive() -> None:
    print(f"[INFO] Drive에 venv tar.gz 저장 중: {VLLM_ARCHIVE}")
    tmp_archive = VLLM_ARCHIVE.with_name(VLLM_ARCHIVE.name + ".partial")
    if tmp_archive.exists():
        tmp_archive.unlink()
    subprocess.run(
        ["tar", "-czf", str(tmp_archive), "-C", "/content", "vllm_env"],
        check=True,
    )
    tmp_archive.replace(VLLM_ARCHIVE)
    VLLM_ARCHIVE_META.write_text(
        json.dumps(
            {
                "python_tag": _python_tag,
                "python": sys.version.split()[0],
                "vllm_spec": VLLM_VERSION_SPEC,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    print("[OK] Drive venv 캐시 저장 완료")


def _prepare_vllm_venv() -> None:
    if _vllm_venv_ok(VLLM_VENV):
        print(f"[SKIP] 로컬 venv 재사용: {VLLM_VENV}")
        return
    if VLLM_VENV.exists():
        shutil.rmtree(VLLM_VENV)
    if _vllm_archive_usable():
        print(f"[INFO] Drive tar.gz에서 venv 복원 중: {VLLM_ARCHIVE}")
        subprocess.run(["tar", "-xzf", str(VLLM_ARCHIVE), "-C", "/content"], check=True)
        if _vllm_venv_ok(VLLM_VENV):
            print("[OK] Drive tar 복원 완료")
            return
        print("[WARN] 복원한 venv가 이 런타임과 맞지 않습니다. 다시 설치합니다.")
        shutil.rmtree(VLLM_VENV, ignore_errors=True)
    print("[INFO] vLLM 격리 가상환경 설치 중 (pip 캐시는 Drive)...")
    subprocess.run(["pip", "install", "-q", "virtualenv"], check=True, env=_pip_env)
    subprocess.run(["virtualenv", str(VLLM_VENV)], check=True)
    subprocess.run(
        [str(VLLM_VENV / "bin" / "pip"), "install", "-q", "--cache-dir", str(PIP_CACHE_DIR), VLLM_VERSION_SPEC],
        check=True,
        env=_pip_env,
    )
    if not _vllm_venv_ok(VLLM_VENV):
        raise RuntimeError("vLLM 가상환경 설치 후 python/vllm 바이너리가 없습니다.")
    print("[OK] vLLM 격리 가상환경 준비 완료:", VLLM_VENV)
    try:
        _save_vllm_archive()
    except Exception as exc:
        print(f"[WARN] Drive tar 저장 실패(이번 세션 실행은 가능): {type(exc).__name__}: {exc}")


VLLM_LOG_PATH = Path("/tmp/vllm_serve.log")
VLLM_SERVED_MODEL = None


def _vllm_log_tail(n: int = 80) -> str:
    if not VLLM_LOG_PATH.is_file():
        return "(로그 파일 없음)"
    try:
        lines = VLLM_LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()
    except Exception as exc:
        return f"(로그 읽기 실패: {type(exc).__name__}: {exc})"
    return "\n".join(lines[-n:]) if lines else "(로그 비어 있음)"


def _resolve_vllm_model() -> str | None:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{VLLM_PORT}/v1/models", timeout=3) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
    except (urllib.error.URLError, TimeoutError, json.JSONDecodeError, OSError):
        return None
    models = [
        item.get("id")
        for item in (payload.get("data") or [])
        if isinstance(item, dict) and item.get("id")
    ]
    return models[0] if models else None


def _wait_vllm_ready(timeout: float = 300.0) -> str:
    deadline = time.time() + timeout
    last_log = 0.0
    while time.time() < deadline:
        proc = globals().get("vllm_process")
        if proc is not None and getattr(proc, "poll", None) is not None and proc.poll() is not None:
            raise RuntimeError(
                f"vLLM 프로세스가 종료되었습니다 (code={proc.returncode}). 로그:\n{_vllm_log_tail()}"
            )
        model_id = _resolve_vllm_model()
        if model_id:
            return model_id
        elapsed = timeout - (deadline - time.time())
        if elapsed - last_log >= 15:
            print(f"[INFO] vLLM /v1/models 대기 중... {int(elapsed)}s / {int(timeout)}s")
            last_log = elapsed
        time.sleep(1.0)
    raise RuntimeError(
        f"vLLM /v1/models가 {int(timeout)}초 안에 준비되지 않았습니다. 로그:\n{_vllm_log_tail()}"
    )

if _is_port_open(VLLM_PORT):
    print(f"[SKIP] 포트 {VLLM_PORT} 이미 사용 중 — 기존 vLLM 서버 재사용")
else:
    _prepare_vllm_venv()
    print("[INFO] vLLM 서버 기동 중. 포트가 열려도 모델 로드가 끝날 때까지 기다립니다.")
    VLLM_LOG_PATH.write_text("", encoding="utf-8")
    _vllm_log_f = open(VLLM_LOG_PATH, "ab", buffering=0)
    vllm_process = subprocess.Popen([
        str(VLLM_VENV / "bin" / "vllm"), "serve", str(MERGED_MODEL_PATH),
        "--port", str(VLLM_PORT),
        "--dtype", "bfloat16",
        "--served-model-name", "qwen3-4b-instruct-2507-merged",
        "--gpu-memory-utilization", str(globals().get("VLLM_GPU_MEMORY_UTILIZATION", 0.30)),
        "--max-model-len", str(int(globals().get("VLLM_MAX_MODEL_LEN", 8192))),
    ], stdout=_vllm_log_f, stderr=subprocess.STDOUT)
VLLM_SERVED_MODEL = _wait_vllm_ready(300.0)
print(f"[OK] vLLM 모델 준비: {VLLM_SERVED_MODEL} (포트 {VLLM_PORT})")


[SKIP] 로컬 venv 재사용: /content/vllm_env
[INFO] vLLM 서버 기동 중. 포트가 열려도 모델 로드가 끝날 때까지 기다립니다.
[INFO] vLLM /v1/models 대기 중... 15s / 300s
[OK] vLLM 모델 준비: qwen3-4b-instruct-2507-merged (포트 8001)


In [19]:
# ============================================================
# [셀 20/26] vLLM — generate_with_qwen 정의
# - 반환은 (answer, metrics)이다. generate_rag_answer·rag_cache·/ask가 호출한다.
# - /analyze는 XAI SDPA 모델을 사용한다.
# - 답변용 HF 모델이 메모리에 있으면 vLLM, XAI와 VRAM을 나눠 갖게 되므로 해제한다.
#   SKIP_ANSWER_MODEL_LOAD=True이면 로드되지 않았으므로 이 단계는 건너뛴다.
# ============================================================

import subprocess
import time

try:
    from openai import OpenAI
except ImportError:
    subprocess.run(["pip", "install", "-q", "openai"], check=True)
    from openai import OpenAI

if "VLLM_PORT" not in globals() or "MERGED_MODEL_PATH" not in globals():
    raise RuntimeError("셀 18, 19 실행 필요")

# 답변용 HF 모델이 올라가 있으면 GPU 메모리를 반환한다.
if (
    "generation_model" in globals()
    and generation_model is not None
    and not bool(globals().get("HYBRID_RUNTIME_SHARED_MODEL", False))
):
    print("[INFO] generation_model GPU 메모리 해제 중...")
    del globals()["generation_model"]
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print("[OK] generation_model 해제 완료")
    if torch.cuda.is_available():
        free_b, total_b = torch.cuda.mem_get_info()
        print(f"[INFO] 해제 후 GPU 여유 메모리: {free_b / 1e9:.2f} GB / {total_b / 1e9:.2f} GB")
else:
    print("[SKIP] generation_model이 없거나 xai_model과 공유 중 — 해제하지 않음")
# --------------------------------------------------------

vllm_client = OpenAI(base_url=f"http://127.0.0.1:{VLLM_PORT}/v1", api_key="not-needed", timeout=300.0)

def generate_with_qwen(query: str, context_text: str) -> tuple:
    """vLLM OpenAI 호환 서버로 답변을 생성한다. 반환은 (answer, metrics)."""
    messages = build_messages_for_generation(query, context_text)
    do_sample = True
    temperature = GEN_TEMPERATURE
    extra_body = {"repetition_penalty": GEN_REPETITION_PENALTY}

    start_time = time.perf_counter()
    model_id = str(globals().get("VLLM_SERVED_MODEL") or MERGED_MODEL_PATH)
    response = vllm_client.chat.completions.create(
        model=model_id,
        messages=messages,
        max_tokens=MAX_NEW_TOKENS,
        temperature=temperature,
        top_p=GEN_TOP_P,
        extra_body=extra_body,
    )
    generation_seconds = time.perf_counter() - start_time

    answer = (response.choices[0].message.content or "").strip()
    generated_token_count = response.usage.completion_tokens if response.usage else 0
    tps = generated_token_count / generation_seconds if generation_seconds > 0 else 0.0
    finish_reason = response.choices[0].finish_reason

    print("========== LLM generation metrics (vLLM) ==========")
    print(f"generation_seconds: {generation_seconds:.2f} sec")
    print(f"generated_tokens  : {generated_token_count}")
    print(f"TPS               : {tps:.2f} tokens/sec")
    print("====================================================")

    generation_metrics = {
        "generation_ms": round(generation_seconds * 1000, 2),
        "generated_tokens": generated_token_count,
        "generation_tps": round(tps, 2),
        "generation_tps_definition": "completion_tokens / vLLM HTTP wall seconds; includes prefill and network",
        "finish_reason": finish_reason,
        "do_sample": do_sample,
        "hybrid_attention": False,
        "answer_attention_backend": "vllm",
        "xai_attention_backend": globals().get("XAI_ATTENTION_DETAIL", "unknown"),
        "answer_lora_status": "merged_before_vllm_load",
        "xai_lora_status": globals().get("XAI_LORA_STATUS", "unknown"),
        "answer_device": f"vllm:127.0.0.1:{VLLM_PORT}",
    }
    return answer, generation_metrics

print("[OK] generate_with_qwen()이 vLLM 호출로 정의되었습니다.")
if "generate_rag_answer" in globals():
    _rag_ns = generate_rag_answer.__globals__
    _rag_ns["generate_with_qwen"] = generate_with_qwen
    _rag_ns["vllm_client"] = vllm_client
    if "VLLM_SERVED_MODEL" in globals():
        _rag_ns["VLLM_SERVED_MODEL"] = VLLM_SERVED_MODEL
    for _name in ("_empty_ask_metrics", "_print_pipeline_metrics", "rag_cache",
                  "make_retrieval_debug_item", "build_retrieval_debug"):
        if _name in globals():
            _rag_ns[_name] = globals()[_name]

def _warmup_inference() -> None:
    """첫 질문 전에 임베딩, reranker, vLLM 커널만 한 번씩 실행한다.
    검색, 대화 저장, 화면 기록은 하지 않는다.
    """
    started = time.perf_counter()
    print("[INFO] 추론 워밍업 시작. 채팅 기록에는 남지 않습니다.")
    try:
        if "embed_model" in globals():
            embed_model.encode(
                ["ping"],
                convert_to_numpy=True,
                normalize_embeddings=True,
            )
        if "reranker_model" in globals() and "reranker_tokenizer" in globals():
            device = next(reranker_model.parameters()).device
            batch = reranker_tokenizer(
                ["ping"],
                ["pong"],
                padding=True,
                truncation=True,
                max_length=32,
                return_tensors="pt",
            )
            batch = {key: value.to(device) for key, value in batch.items()}
            with torch.no_grad():
                reranker_model(**batch)
        model_id = str(globals().get("VLLM_SERVED_MODEL") or MERGED_MODEL_PATH)
        vllm_client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": "ping"}],
            max_tokens=8,
            temperature=0,
        )
    except Exception as exc:
        print(f"[WARN] 추론 워밍업 실패. 서버 기동은 유지합니다: {type(exc).__name__}: {exc}")
        return
    print(f"[OK] 추론 워밍업 완료: {time.perf_counter() - started:.1f}s")


_warmup_inference()


[SKIP] generation_model이 없거나 xai_model과 공유 중 — 해제하지 않음
[OK] generate_with_qwen()이 vLLM 호출로 정의되었습니다.
[INFO] 추론 워밍업 시작. 채팅 기록에는 남지 않습니다.
[OK] 추론 워밍업 완료: 0.3s


In [20]:
# ============================================================
# [셀 21/26] Frontend 런타임 준비 (1/3) — 경로·환경 확인
# - https://docs.python.org/ko/3/library/shutil.html
# - GitHub 원본은 수정하지 않고 /content/runtime_app 만 패치(깃헙에서 app 폴더에 해당)
# - repo의 colab/*.ipynb 는 clone만 되고 실행·복사되지 않아 충돌하지 않음
# ============================================================

import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/xai-capstone-2026/XAI_Capstone_Project.git"
REPO_DIR = Path("/content/XAI_Capstone_Project")
RUNTIME_APP = Path("/content/runtime_app")
FRONTEND_PORT = 8000
RAG_API_BASE = "http://127.0.0.1:9000"

print("REPO_URL   :", REPO_URL)
print("REPO_DIR   :", REPO_DIR)
print("RUNTIME_APP:", RUNTIME_APP)

git_check = subprocess.run(
    ["git", "--version"],
    capture_output=True,
    text=True,
)
print("git 확인  :", (git_check.stdout or git_check.stderr).strip())
if git_check.returncode != 0:
    raise RuntimeError("git 명령을 사용할 수 없습니다.")

disk = shutil.disk_usage("/content")
free_gb = disk.free / (1024**3)
print(f"/content 여유 디스크: {free_gb:.2f} GB")
if free_gb < 1.0:
    print("[WARN] 디스크 여유가 1GB 미만 남았습니다.")


REPO_URL   : https://github.com/xai-capstone-2026/XAI_Capstone_Project.git
REPO_DIR   : /content/XAI_Capstone_Project
RUNTIME_APP: /content/runtime_app
git 확인  : git version 2.43.0
/content 여유 디스크: 153.05 GB


In [21]:
# ============================================================
# [셀 22/26] Frontend 런타임 준비 (2/3) — git clone + 성공 여부 확인
# ============================================================

import shutil
import subprocess
from pathlib import Path

if "REPO_DIR" not in globals():
    raise RuntimeError("21셀부터 실행 필요")

if REPO_DIR.exists():
    print(f"기존 폴더 제거: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

print("git sparse clone 시작 (app/ 디렉터리만)...")
clone_result = subprocess.run(
    ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", REPO_URL, str(REPO_DIR)],
    capture_output=True,
    text=True,
)
if clone_result.returncode == 0:
    sparse_result = subprocess.run(
        ["git", "sparse-checkout", "set", "app"],
        cwd=str(REPO_DIR),
        capture_output=True,
        text=True,
    )
    if sparse_result.returncode != 0:
        print("[경고] sparse-checkout 실패, 전체 clone으로 재시도합니다.")
        shutil.rmtree(REPO_DIR)
        clone_result = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            capture_output=True,
            text=True,
        )
    else:
        print("[OK] sparse-checkout: app/ 만 받았습니다.")

print("=" * 60)
print("git exit code:", clone_result.returncode)
if clone_result.stdout.strip():
    # exit code: 0이면 정상
    print("--- stdout ---")
    print(clone_result.stdout.strip()[-2000:])
if clone_result.stderr.strip():
    # 'stderr'가 출력되어도 정상, 일반적인 진행 상황이나 안내 문구도 stderr로 출력됨
    print("--- stderr ---")
    print(clone_result.stderr.strip()[-2000:])
print("=" * 60)

if clone_result.returncode != 0:
    raise RuntimeError(
        f"git clone 실패 (exit={clone_result.returncode}). "
        "위 stderr를 확인 필요(네트워크, repo URL, 디스크)"
    )

if not (REPO_DIR / ".git").is_dir():
    raise RuntimeError(f"clone 후 .git 없음: {REPO_DIR}")

app_main = REPO_DIR / "app" / "main.py"
if not app_main.is_file():
    raise RuntimeError(f"clone 후 app/main.py 없음: {app_main}")

repo_top = sorted(p.name for p in REPO_DIR.iterdir())
print("[OK] git clone 성공")
print("repo 최상위:", repo_top)

colab_dir = REPO_DIR / "colab"
if colab_dir.is_dir():
    nb_count = len(list(colab_dir.glob("*.ipynb")))
    print(f"repo/colab/*.ipynb: {nb_count}개 (참고용, 이 노트북과 자동 충돌하지 않음)")
else:
    print("repo/colab/ 폴더 없음")


기존 폴더 제거: /content/XAI_Capstone_Project
git sparse clone 시작 (app/ 디렉터리만)...
[OK] sparse-checkout: app/ 만 받았습니다.
git exit code: 0
--- stderr ---
Cloning into '/content/XAI_Capstone_Project'...
[OK] git clone 성공
repo 최상위: ['.git', '.gitignore', 'Readme.md', 'app', 'requirements.txt']
repo/colab/ 폴더 없음


In [22]:
# ============================================================
# [셀 23/26] Frontend 런타임 준비 (3/3) — app/ 복사 + main.py 패치
# ============================================================

import shutil
from pathlib import Path

if "REPO_DIR" not in globals() or "RUNTIME_APP" not in globals():
    raise RuntimeError("21, 22셀부터 실행 필요")

app_src = REPO_DIR / "app"
if not (app_src / "main.py").is_file():
    raise RuntimeError(f"app/main.py 없음. 22셀 git clone 셀을 다시 실행 필요: {app_src}")

if RUNTIME_APP.exists():
    print(f"기존 runtime_app 제거: {RUNTIME_APP}")
    shutil.rmtree(RUNTIME_APP)

shutil.copytree(app_src, RUNTIME_APP)
print(f"[OK] runtime_app 복사 완료: {RUNTIME_APP}")
print("복사된 항목:", sorted(p.name for p in RUNTIME_APP.iterdir()))

main_path = RUNTIME_APP / "main.py"
main_text = main_path.read_text(encoding="utf-8")

PATCHES = [
    (
        "from app.services.conversation_store import ConversationStore",
        "from services.conversation_store import ConversationStore",
    ),
    ('StaticFiles(directory="app/static")', 'StaticFiles(directory="static")'),
    ('Jinja2Templates(directory="app/templates")', 'Jinja2Templates(directory="templates")'),
    (
        'COLAB_ASK_URL = "https://api.kr-welfare-xai.com/ask"',
        f'COLAB_ASK_URL = "{RAG_API_BASE}/ask"',
    ),
    (
        'COLAB_ANALYZE_URL = "https://api.kr-welfare-xai.com/analyze"',
        f'COLAB_ANALYZE_URL = "{RAG_API_BASE}/analyze"',
    ),
    (
        "timeout=60.0,",
        "timeout=300.0,",
    ),
    (
        '"query_summary": query_summary,\n'
        '    }',
        '"query_summary": query_summary,\n'
        '        "word_scores": _build_word_scores(captum_raw),\n'
        '    }',
    ),
    (
        '"query_items": [],\n'
        '        "raw": {',
        '"query_items": [],\n'
        '        "word_scores": [],\n'
        '        "raw": {',
    ),
    (
        '"query_summary": xai.get(\"query_summary\"),\n'
        '                },\n',
        '"query_summary": xai.get(\"query_summary\"),\n'
        '                    \"word_scores\": xai.get(\"word_scores\", []),\n'
        '                },\n',
    ),
    (
        'def _normalize_captum_result(captum_raw: dict | None) -> dict:\n'
        '    if not isinstance(captum_raw, dict):',
        'def _normalize_captum_result(captum_raw: dict | None) -> dict:\n'
        '    if captum_raw.get("error"):\n'
        '        message = captum_raw.get("message") or "Captum 분석 중 오류가 발생했습니다."\n'
        '        result = _make_xai_error(message)\n'
        '        word_scores = _build_word_scores(captum_raw)\n'
        '        if word_scores:\n'
        '            result["word_scores"] = word_scores\n'
        '            result["status"] = "partial"\n'
        '        return result\n\n'
        '    if not isinstance(captum_raw, dict):',
    ),
]

if "_build_word_scores" not in main_text:
    helper = (
        "\n\n\ndef _build_word_scores(captum_raw: dict | None, top_n: int = 3) -> list:\n"
        "    if not isinstance(captum_raw, dict):\n        return []\n"
        "    positions = captum_raw.get(\"query_positions\") or []\n"
        "    if not isinstance(positions, list) or not positions:\n        return []\n"
        "    ranked = sorted([p for p in positions if isinstance(p, dict)], key=lambda p: _safe_float(p.get(\"percent\")), reverse=True)[:top_n]\n"
        "    word_scores = []\n"
        "    for pos in ranked:\n"
        "        percent = _safe_float(pos.get(\"percent\"))\n"
        "        if percent <= 0:\n            continue\n"
        "        start, end = pos.get(\"start\"), pos.get(\"end\")\n"
        "        if not isinstance(start, int) or not isinstance(end, int) or start < 0 or end <= start:\n            continue\n"
        "        word_scores.append({\"start\": start, \"end\": end, \"score\": percent / 100.0, \"label\": str(pos.get(\"name\") or \"\")})\n"
        "    return word_scores\n"
    )
    anchor = "def _normalize_captum_result(captum_raw: dict | None) -> dict:"
    if anchor in main_text:
        main_text = main_text.replace(anchor, helper + anchor, 1)
    else:
        print("[WARN] _build_word_scores 삽입 실패")

for old, new in PATCHES:
    if old not in main_text:
        print(f"[WARN] 패치 대상 없음: {old[:60]}...")
    main_text = main_text.replace(old, new)

main_path.write_text(main_text, encoding="utf-8")
print("[OK] runtime_app main.py 패치 완료 (RAG API → 127.0.0.1:9000)")

chat_path = RUNTIME_APP / "static" / "js" / "chat.js"
if chat_path.is_file():
    chat_text = chat_path.read_text(encoding="utf-8")
    if "function renderBotText" not in chat_text:
        chat_text = chat_text.replace(
            "content.className = 'markdown-body';",
            "content.className = 'bot-plain';",
            1,
        )
        chat_text = chat_text.replace(
            "content.innerHTML = renderMarkdown(text || '');",
            "content.innerHTML = renderBotText(text || '');",
            1,
        )
        chat_text = chat_text.replace(
            "textNode.className = 'markdown-body';",
            "textNode.className = 'bot-plain';",
            1,
        )
        chat_text = chat_text.replace(
            "textNode.innerHTML = renderMarkdown(text);",
            "textNode.innerHTML = renderBotText(text);",
            1,
        )
        bot_helper = (
            "function renderBotText(text) {\n"
            "    return escapeHtml(text || '').replace(/\\n/g, '<br>');\n"
            "}\n\n"
        )
        chat_text = chat_text.replace(
            "function renderMarkdown(text) {",
            bot_helper + "function renderMarkdown(text) {",
            1,
        )
    chat_path.write_text(chat_text, encoding="utf-8")
    print("[OK] runtime_app chat.js word_scores 패치 완료")
else:
    print(f"[WARN] chat.js 없음: {chat_path}")

css_path = RUNTIME_APP / "static" / "css" / "chat.css"
if css_path.is_file():
    css_text = css_path.read_text(encoding="utf-8")
    if ".bot-plain" not in css_text:
        css_text = css_text.replace(
            ".markdown-body {\n    line-height: 1.7;\n}",
            ".markdown-body {\n    line-height: 1.7;\n}\n\n.bot-plain {\n    white-space: pre-wrap;\n    word-break: break-word;\n    line-height: 1.7;\n}",
            1,
        )
        css_path.write_text(css_text, encoding="utf-8")
        print("[OK] runtime_app chat.css bot-plain 패치 완료")
else:
    print(f"[WARN] chat.css 없음: {css_path}")


기존 runtime_app 제거: /content/runtime_app
[OK] runtime_app 복사 완료: /content/runtime_app
복사된 항목: ['main.py', 'services', 'static', 'templates']
[WARN] 패치 대상 없음: def _normalize_captum_result(captum_raw: dict | None) -> dic...
[OK] runtime_app main.py 패치 완료 (RAG API → 127.0.0.1:9000)
[OK] runtime_app chat.js word_scores 패치 완료
[OK] runtime_app chat.css bot-plain 패치 완료


In [23]:
# ============================================================
# [셀 24/26] RAG/XAI FastAPI 서버 (포트 9000, host=127.0.0.1)
# - /ask, /analyze: RAG·LLM·Captum 처리
# - 셀 25 runtime_app(8000)이 http://127.0.0.1:9000으로 호출
# ============================================================

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import uvicorn
import nest_asyncio
import traceback
import threading
import time
from typing import Optional


def _captum_token_hash(ids):
    import hashlib
    import json
    return hashlib.sha256(json.dumps(ids, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()


def _log_step(msg: str) -> None:
    print(msg, flush=True)


nest_asyncio.apply()

if "generate_rag_answer" not in globals():
    raise RuntimeError("셀 17 실행 필요")

_log_step("[셀 24/26] RAG API 9000 — 1/3 FastAPI 앱 생성...")

RAG_API_PORT = 9000
rag_app = FastAPI()

# 동시 /ask·/analyze 요청 시 GPU OOM 방지 (셀 17 RLock 재사용)
if "generation_lock" not in globals():
    generation_lock = threading.RLock()


class ChatRequest(BaseModel):
    query: str
    run_id: Optional[str] = None


@rag_app.get("/health")
def rag_health():
    return {
        "status": "ok",
        "service": "rag-xai",
        "port": RAG_API_PORT,
    }


@rag_app.get("/")
def root():
    return {
        "status": "ok",
        "message": "XAI RAG Reranker API is running.",
        "port": RAG_API_PORT,
    }


@rag_app.post("/ask")
def ask_api(request: ChatRequest):
    try:
        query = (request.query or "").strip()
        print(f"\n[FastAPI] 질문 도착: {query}")

        if not query:
            return {
                "answer": "질문 내용이 비어 있습니다. 궁금한 정책이나 상황을 입력해 주세요.",
                "retrieval_debug": {
                    "before_rerank": [],
                    "after_rerank": [],
                },
            }

        with generation_lock:
            result = generate_rag_answer(query)

        return result

    except Exception:
        error_trace = traceback.format_exc()
        print(f"[FastAPI] /ask 에러 발생:\n{error_trace}")

        return JSONResponse(status_code=500, content={
            "answer": "서버에서 답변을 생성하는 중 오류가 발생했습니다.",
            "error": True,
            "detail": error_trace[-2000:],
        })


@rag_app.post("/analyze")
def analyze_api(request: ChatRequest):
    try:
        return analyze_with_captum(request.query)
    except Exception:
        traceback.print_exc()
        return JSONResponse(status_code=500, content={
            "error": True, "words": [], "attributions": [], "query_words": [],
            "query_attributions": [], "message": "Captum 분석 중 오류가 발생했습니다."})


_log_step("[셀 24/26] RAG API 9000 — 2/3 uvicorn 기동 시작...")

import socket
import subprocess


def _is_port_open(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.5):
            return True
    except OSError:
        return False


def _kill_port(port: int) -> None:
    if not _is_port_open(port):
        _log_step(f"  · 포트 {port} 비어 있음 — 프로세스 정리 생략")
        return
    _log_step(f"  · 포트 {port} 사용 중 — pkill 시도 (최대 3초)")
    try:
        subprocess.run(
            ["pkill", "-f", f"uvicorn.*--port {port}"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=3,
        )
    except (subprocess.TimeoutExpired, FileNotFoundError, OSError):
        pass
    time.sleep(0.3)


def _wait_port(port: int, timeout: float = 15.0) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        if _is_port_open(port):
            return True
        time.sleep(0.3)
    return False


def _start_uvicorn_thread(app, port: int, thread_name: str) -> threading.Thread:
    def _run():
        uvicorn.run(app, host="127.0.0.1", port=port, log_level="warning")

    t = threading.Thread(target=_run, name=thread_name, daemon=True)
    t.start()
    return t


rag_server_thread = None
reuse_existing = False
if _is_port_open(RAG_API_PORT):
    try:
        import urllib.request

        with urllib.request.urlopen(
            f"http://127.0.0.1:{RAG_API_PORT}/health", timeout=2
        ) as resp:
            if resp.status == 200:
                reuse_existing = True
                _log_step(f"  · 포트 {RAG_API_PORT} 기존 서버 재사용 (/health 200)")
    except Exception:
        pass

if not reuse_existing:
    _log_step("  · 포트 정리...")
    _kill_port(RAG_API_PORT)
    _log_step("  · uvicorn 스레드 시작...")
    rag_server_thread = _start_uvicorn_thread(
        rag_app, RAG_API_PORT, "rag_uvicorn_thread"
    )
    _log_step("  · 포트 9000 응답 대기...")

if reuse_existing or _wait_port(RAG_API_PORT):
    _log_step(f"[OK] RAG/XAI API 수신 확인 (포트 {RAG_API_PORT})")
    try:
        import urllib.request

        with urllib.request.urlopen(
            f"http://127.0.0.1:{RAG_API_PORT}/health", timeout=5
        ) as resp:
            _log_step(f"   GET /health → {resp.status}")
    except Exception as exc:
        _log_step(f"   /health 확인 실패(무시 가능): {exc}")
else:
    _log_step(
        f"[FAIL] 포트 {RAG_API_PORT} 15초 내 응답 없음 — 런타임 재시작 후 셀 24만 다시 실행"
    )
    if rag_server_thread is not None and not rag_server_thread.is_alive():
        _log_step("   uvicorn 스레드가 즉시 종료됨. 셀 17 실행 여부를 확인 필요")

_log_step("[셀 24/26] RAG API 9000 — 3/3 Captum 함수 정의 중...")

_kiwi_instance = None


def _get_kiwi():
    global _kiwi_instance
    if _kiwi_instance is None:
        from kiwipiepy import Kiwi
        _log_step("[RAG API] Kiwi 로딩 중...")
        _kiwi_instance = Kiwi()
        _log_step("[RAG API] Kiwi 로드 완료")
    return _kiwi_instance


def extract_important_words(text: str) -> list:
    """
    한글 형태소 분석기(Kiwi)를 사용하여 질문에서 핵심 키워드(명사, 숫자, 외국어 등)만 추출합니다.
    동사, 형용사, 의존명사, 조사 등은 RAG 검색 의도 파악에서 배제됩니다.
    연속된 명사(예: 취업 + 준비 + 생)는 하나의 복합명사로 병합하여 마스킹 단위를 보존합니다.
    """
    return [position["name"] for position in extract_query_word_positions(text)]


def extract_query_word_positions(text: str) -> list:
    if not text:
        return []
    tokens = _get_kiwi().tokenize(text)
    # NNG(일반명사), NNP(고유명사), NR(수사), SN(숫자), SL(알파벳), SH(한자), XR(어근), XPN(체언접두사), XSN(명사파생접미사)
    valid_tags = {'NNG', 'NNP', 'NR', 'SN', 'SL', 'SH', 'XR', 'XPN', 'XSN'}
    positions = []
    current_word = ""
    current_start = None
    last_end = -1
    for token in tokens:
        if token.tag in valid_tags:
            # 이전 토큰과 공백 없이 바로 이어지는 경우 (복합명사 병합)
            if current_word and token.start == last_end:
                current_word += token.form
            else:
                if current_word:
                    positions.append({"name": current_word, "start": current_start, "end": last_end})
                current_word = token.form
                current_start = token.start
            last_end = token.start + token.len
        else:
            if current_word:
                positions.append({"name": current_word, "start": current_start, "end": last_end})
                current_word = ""
                current_start = None
            last_end = -1
    if current_word:
        positions.append({"name": current_word, "start": current_start, "end": last_end})
    return positions


def get_xai_model_device(active_model=None):
    active_model = active_model or globals().get("model")
    try:
        return next(active_model.parameters()).device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _release_cuda_cache() -> None:
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def analyze_with_captum(query):
    with generation_lock:
        try:
            return _analyze_v2_original(query)
        finally:
            _release_cuda_cache()

def _analyze_v2_original(query: str):
    import torch
    from captum.attr import FeatureAblation

    analyze_start = time.perf_counter()
    cache_hit = False

    xai_model = globals().get("model")
    if xai_model is None:
        raise RuntimeError("XAI 모델이 없습니다. 셀 08을 먼저 실행하세요.")
    if hasattr(xai_model, "config"):
        xai_model.config.use_cache = False

    # 문서 단위로 masking 처리 후 타겟 문장 도출 확률을 비교
    # 사용 captum 알고리즘: FeatureAblation
    # Attribution(x_i) = P(y | x) - P(y | x_{\setminus i})
    # (특정 문서 또는 질문의 기여도) = (질문 query + 전체 문서로 타겟 문장 도출 확률) - (특정 문서 또는 질문 masking 후 타겟 문장 도출 확률)


    query = (query or "").strip()
    if not query:
        return {
            "words": [], "attributions": [],
            "query_words": [], "query_attributions": [],
            "message": "질문 내용이 비어 있습니다."
        }

    print("\n[Captum XAI 분석 시작]")

    cached_data = rag_cache.get(query)

    # RAG cache 확인
    if cached_data:
        cache_hit = True
        print("[Captum XAI] 기존 캐시에서 문맥과 답변을 불러옵니다.")
        context_text = cached_data["context_text"]
        selected_policies = cached_data["selected_policies"]
        answer = cached_data["answer"]
    # 없을 시 질문 query 인자로 RAG 검색 및 reranker로 답변에 사용된 문서 생성
    else:
        print("[Captum XAI] 캐시가 없습니다. RAG 파이프라인을 새로 실행합니다.")
        candidates = retrieve_candidates(query)
        if not candidates:
            return {
                "words": [], "attributions": [],
                "query_words": [], "query_attributions": [],
                "answer": "검색 결과가 없습니다."
            }

        reranked_items = rerank_candidates(query, candidates)
        if not reranked_items:
            return {
                "words": [], "attributions": [],
                "query_words": [], "query_attributions": [],
                "answer": "재정렬 결과가 없습니다."
            }

        context_result = build_final_context(reranked_items)
        context_text = context_result["context_text"]
        selected_policies = context_result["selected_policies"]

        if not selected_policies:
            return {
                "words": [], "attributions": [],
                "query_words": [], "query_attributions": [],
                "answer": "최종 선택된 정책이 없습니다."
            }

        with generation_lock:
            answer, _ = generate_with_qwen(query, context_text)

        if not answer:
            return {
                "words": [], "attributions": [],
                "query_words": [], "query_attributions": [],
                "answer": "답변 생성 실패"
            }

    # 답변에 사용된 프롬프트 토큰화
    messages = build_messages_for_generation(query, context_text)
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    try:
        inputs = tokenizer(input_text, return_tensors="pt", return_offsets_mapping=True)
        offset_mapping = inputs.get("offset_mapping")
        if offset_mapping is not None:
            offset_mapping = offset_mapping[0].tolist()
    except Exception:
        inputs = tokenizer(input_text, return_tensors="pt")
        offset_mapping = None

    xai_device = get_xai_model_device(xai_model)
    input_ids = inputs["input_ids"].to(xai_device)
    target_ids = tokenizer(answer, return_tensors="pt", add_special_tokens=False)["input_ids"].to(xai_device)


    # masking 동적 생성
    # feature_mask는 input_ids와 똑같은 길이를 가지며, 각 토큰이 속한 그룹(문서 번호)을 기록
    feature_mask = torch.zeros_like(input_ids)

    # 각 문서의 시작 위치 탐색
    doc_positions = []
    for idx, policy in enumerate(selected_policies, 1):
        header = f"[문서 {idx}]"
        start_char = input_text.find(header)
        doc_positions.append({
            "id": idx,
            "start": start_char,
            "name": f"문서 {idx}: {policy.get('policy_name', '')[:20]}..."
        })

    # 질문 query 시작 위치 탐색
    q_start = input_text.find("[사용자 질문]")
    doc_positions.append({
        "id": len(selected_policies) + 1,
        "start": q_start,
        "name": "사용자 질문"
    })

    input_ids_list = input_ids[0].tolist()

    # 전체 토큰 탐색, 문서 별 mask 번호 부여 및 문서에 속하지 않는 영역 처리(시스템 프롬프트 등)
    if offset_mapping is not None:
        for t_idx, (t_start, t_end) in enumerate(offset_mapping):
            if t_start == t_end:
                continue
            assigned_id = 0
            t_mid = (t_start + t_end) / 2
            for i in range(len(doc_positions)):
                start = doc_positions[i]["start"]
                end = doc_positions[i+1]["start"] if i+1 < len(doc_positions) else len(input_text)

                if start <= t_mid <= end:
                    assigned_id = doc_positions[i]["id"]
                    break
            feature_mask[0, t_idx] = assigned_id
    else:
        current_char_len = 0

        # 전체 토큰 탐색, 문서 별 mask 번호 부여
        for t_idx, tid in enumerate(input_ids_list):
            t_str = tokenizer.decode([tid], skip_special_tokens=False)
            current_char_len += len(t_str)

            assigned_id = 0
            for i in range(len(doc_positions)):
                start = doc_positions[i]["start"]
                end = doc_positions[i+1]["start"] if i+1 < len(doc_positions) else len(input_text)

                if start <= current_char_len <= end:
                    assigned_id = doc_positions[i]["id"]
                    break

            feature_mask[0, t_idx] = assigned_id

    elements = [position["name"] for position in doc_positions]

    # ------------------------------------------------------------
    # 답변 토큰 구간의 logits만 계산하는 공용 헬퍼
    # ------------------------------------------------------------
    # 전체 위치 logits는 [배치, 시퀀스, 151936]이다.
    # 기여도에 쓰는 구간은 답변 토큰뿐이므로 hidden state에서 그 구간만 lm_head에 넣는다.
    # 분리 접근이 불가능하면 전체 logits에서 같은 구간을 자른다.
    _xai_base = getattr(xai_model, "model", None)
    _xai_head = getattr(xai_model, "lm_head", None)
    _use_sliced_head = _xai_base is not None and _xai_head is not None
    if not _use_sliced_head:
        print("[WARN] base model/lm_head 분리 접근 불가 — 전체 logits 경로로 동작한다")

    def _target_logits(full_ids, prompt_len, target_len):
        lo = prompt_len - 1
        hi = prompt_len + target_len - 1
        if not _use_sliced_head:
            return xai_model(input_ids=full_ids).logits[:, lo:hi, :]
        hidden = _xai_base(input_ids=full_ids).last_hidden_state
        return _xai_head(hidden[:, lo:hi, :])

    # Forward 래퍼 함수
    # masking 단위 특정 문서 pad 후, 정답 도출 확률 산출
    def forward_func(ablated_input_ids):
        # 특정 항목 masking 된 프롬프트
        batch_size = ablated_input_ids.shape[0]
        repeated_target_ids = target_ids.repeat(batch_size, 1)

        # 위 프롬프트와 전체 프롬프트 참고한 모델 생성 답변 포함
        full_input_ids = torch.cat([ablated_input_ids, repeated_target_ids], dim=1)

        # 문서 토큰과 생성 답변 토큰 길이 파악
        seq_len = ablated_input_ids.shape[1]
        target_len = target_ids.shape[1]

        # 생성 답변 토큰 위치의 logit만 계산한다
        shift_logits = _target_logits(full_input_ids, seq_len, target_len).contiguous()
        shift_labels = repeated_target_ids.contiguous()

        # pyTorch의 CrossEntropyLoss함수 사용
        # Softmax: logit을 확률로 변환 / Cross Entropy: 예측 확률과 실제 정답 간 차이 계산
        # [정밀도] 모델은 bfloat16이라 유효숫자가 2~3자리뿐이다.
        # 답변 토큰 수백 개의 로그확률을 그대로 더하면 1000 단위의 큰 값이 되고,
        # FeatureAblation이 그 큰 값 두 개를 빼면서 유효숫자가 날아가 기여도가 정수로 뭉개졌다.
        # 손실 계산부터 float32로 올려 하위권 문서의 해상도를 확보한다.
        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        shift_logits_flat = shift_logits.view(-1, shift_logits.size(-1)).float()
        shift_labels_flat = shift_labels.view(-1)

        loss = loss_fct(shift_logits_flat, shift_labels_flat)
        loss = loss.view(batch_size, target_len)

        # loss 부호 처리 후 프롬프트 별 점수 계산
        log_probs = -loss
        score = log_probs.sum(dim=1)
        return score

    # 6. FeatureAblation 적용
    ablator = FeatureAblation(forward_func)
    baselines = torch.full_like(input_ids, tokenizer.pad_token_id)

    print("Feature Ablation 분석 진행 중...")
    with torch.no_grad():
        with generation_lock:
            attributions = ablator.attribute(
                inputs=input_ids,
                baselines=baselines,
                feature_mask=feature_mask,
                perturbations_per_eval=1 # OOM 방지를 위해 한 번에 1개씩 마스킹
            )

    # 선택된 문서의 기여도를 모두 계산한다. reranker 점수로 기여도를 지우지 않는다.
    attr_scores = []
    for i in range(1, len(doc_positions) + 1):
        mask_idx = (feature_mask == i).nonzero(as_tuple=True)[1]
        if len(mask_idx) > 0:
            score = attributions[0, mask_idx[0]].item()
            attr_scores.append(score)
        else:
            attr_scores.append(0.0)

    # 정규화(백분율)
    # 양수는 해당 부분을 빼면 답변 확률이 떨어지는 근거다.
    # 음수는 빼면 확률이 오르는 경우이므로 백분율에서 0으로 둔다.
    positive_scores = [max(s, 0.0) for s in attr_scores]
    total_score = sum(positive_scores)

    if total_score > 0:
        normalized_scores = [(s / total_score) * 100 for s in positive_scores]
    else:
        normalized_scores = [0.0] * len(attr_scores)

    print("[Captum XAI 분석 종료 - 문서 단위 기여도]")
    for el, raw_sc, norm_sc in zip(elements, attr_scores, normalized_scores):
        print(f"- {el}: 원본 {raw_sc:.4f} -> 정규화 {norm_sc:.2f}%")


    # 질문 쿼리 내부 단어별 기여도 분석
    # 문서는 전체 반영, 질문쿼리 형태소 분해 후 각 단어 masking 후 기여도 산출
    print("\n질문 쿼리 단어별 독립 기여도 분석 준비")
    query_words = extract_important_words(query)

    query_positions = []
    q_text_start = input_text.find(query, q_start)
    current_search_idx = q_text_start

    for idx, word in enumerate(query_words, 1):
        word_start = input_text.find(word, current_search_idx)
        if word_start != -1:
            query_positions.append({
                "id": idx,
                "start": word_start,
                "end": word_start + len(word),
                "name": word
            })
            current_search_idx = word_start + len(word)

    feature_mask_query = torch.zeros_like(input_ids)

    if offset_mapping is not None:
        for t_idx, (t_start, t_end) in enumerate(offset_mapping):
            if t_start == t_end:
                continue
            assigned_id = 0
            for q_pos in query_positions:
                if t_start < q_pos["end"] and t_end > q_pos["start"]:
                    assigned_id = q_pos["id"]
                    break
            feature_mask_query[0, t_idx] = assigned_id
    else:
        current_char_len = 0
        for t_idx, tid in enumerate(input_ids_list):
            t_str = tokenizer.decode([tid], skip_special_tokens=False)
            current_char_len += len(t_str)

            assigned_id = 0
            for q_pos in query_positions:
                if q_pos["start"] <= current_char_len <= q_pos["end"]:
                    assigned_id = q_pos["id"]
                    break
            feature_mask_query[0, t_idx] = assigned_id

    print("질문 단어별 기여도 분석 진행")

    attr_scores_query = []
    with torch.no_grad():
        with generation_lock:
            attributions_query = ablator.attribute(
                inputs=input_ids,
                baselines=baselines,
                feature_mask=feature_mask_query,
                perturbations_per_eval=1,
            )

    for i in range(1, len(query_positions) + 1):
        mask_idx = (feature_mask_query == i).nonzero(as_tuple=True)[1]
        if len(mask_idx) > 0:
            attr_scores_query.append(attributions_query[0, mask_idx[0]].item())
        else:
            attr_scores_query.append(0.0)

    # 정규화(백분율)
    # 문서 기여도와 같은 규칙을 쓴다. 음수는 "답변을 뒷받침하지 않았다"는 뜻이므로
    # abs()로 뒤집지 않고 0으로 절단한다.
    positive_scores_query = [max(s, 0.0) for s in attr_scores_query]
    total_score_query = sum(positive_scores_query)

    if total_score_query > 0:
        normalized_scores_query = [(s / total_score_query) * 100 for s in positive_scores_query]
    else:
        normalized_scores_query = [0.0] * len(query_positions)

    print("[Captum XAI 분석 종료 - 질문 단어별 기여도]")
    for q_pos, raw_sc, norm_sc in zip(query_positions, attr_scores_query, normalized_scores_query):
        print(f"- 단어 '{q_pos['name']}': 원본 {raw_sc:.4f} -> 정규화 {norm_sc:.2f}%")

    analyze_total_ms = round((time.perf_counter() - analyze_start) * 1000, 2)
    print(f"\n[Captum] analyze_total_ms={analyze_total_ms:.2f} cache_hit={cache_hit}")

    return {
        "words": elements,
        "attributions": normalized_scores,
        "query_words": [qp["name"] for qp in query_positions],
        "query_attributions": normalized_scores_query,
        "answer": answer,
        "metrics": {
            "input_ids_hash": _captum_token_hash(input_ids_list),
            "target_ids_hash": _captum_token_hash(target_ids[0].detach().cpu().tolist()),
            "input_tokens": int(input_ids.shape[1]),
            "captum_target_tokens": int(target_ids.shape[1]),
            "captum_ms": analyze_total_ms,
            "analyze_total_ms": analyze_total_ms,
            "cache_hit": cache_hit,
            "query_attr_method": "ablation",
            "xai_attention_backend": globals().get("XAI_ATTENTION_DETAIL", "unknown"),
            "xai_lora_status": globals().get("XAI_LORA_STATUS", "unknown"),
            "xai_use_cache": bool(getattr(getattr(xai_model, "config", None), "use_cache", False)),
        },
    }


print("Captum 준비 완료: 생성 답변 전체 분석, 질문별 최신 캐시")


[셀 24/26] RAG API 9000 — 1/3 FastAPI 앱 생성...
[셀 24/26] RAG API 9000 — 2/3 uvicorn 기동 시작...
  · 포트 정리...
  · 포트 9000 비어 있음 — 프로세스 정리 생략
  · uvicorn 스레드 시작...
  · 포트 9000 응답 대기...
[OK] RAG/XAI API 수신 확인 (포트 9000)
   GET /health → 200
[셀 24/26] RAG API 9000 — 3/3 Captum 함수 정의 중...
Captum 준비 완료: 생성 답변 전체 분석, 질문별 최신 캐시


In [24]:
# ============================================================
# [셀 25/26] Frontend 서버 기동 (포트 8000) — 셀 24 RAG API(9000) 실행 후 수행
# ============================================================

import os
import nest_asyncio
import uvicorn
import importlib.util
from pathlib import Path

nest_asyncio.apply()

RUNTIME_APP = Path("/content/runtime_app")
FRONTEND_PORT = 8000
main_path = RUNTIME_APP / "main.py"

if not main_path.exists():
    raise RuntimeError("runtime_app이 없습니다. 셀 21~23(GitHub clone/패치)을 먼저 실행하세요.")

print("[셀 25/26] Frontend 8000 uvicorn 기동 중...")
os.chdir(RUNTIME_APP)
spec = importlib.util.spec_from_file_location("runtime_main", main_path)
runtime_main = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime_main)

import socket
import threading
import time

if "_kill_port" not in globals():
    def _kill_port(port: int) -> None:
        os.system(f"pkill -f 'uvicorn.*--port {port}' 2>/dev/null || true")
        os.system(f"fuser -k {port}/tcp 2>/dev/null || true")
        time.sleep(1)

    def _wait_port(port: int, timeout: float = 20.0) -> bool:
        deadline = time.time() + timeout
        while time.time() < deadline:
            try:
                with socket.create_connection(("127.0.0.1", port), timeout=1):
                    return True
            except OSError:
                time.sleep(0.5)
        return False

    def _start_uvicorn_thread(app, port: int, thread_name: str) -> threading.Thread:
        _kill_port(port)

        def _run():
            uvicorn.run(app, host="0.0.0.0", port=port, log_level="info")

        t = threading.Thread(target=_run, name=thread_name, daemon=True)
        t.start()
        return t

os.chdir(RUNTIME_APP)
frontend_thread = _start_uvicorn_thread(runtime_main.app, FRONTEND_PORT, "frontend_uvicorn_thread")

if _wait_port(FRONTEND_PORT):
    print(f"[OK] Frontend 수신 확인 (포트 {FRONTEND_PORT})")
    try:
        import httpx
        r = httpx.get(f"http://127.0.0.1:{FRONTEND_PORT}/", timeout=5.0)
        print(f"   GET / → {r.status_code}")
    except Exception as exc:
        print(f"   GET / 확인 실패: {exc}")
else:
    print(f"[FAIL] 포트 {FRONTEND_PORT} 응답 없음")


[셀 25/26] Frontend 8000 uvicorn 기동 중...
[OK] Frontend 수신 확인 (포트 8000)
   GET / → 200


In [25]:
# ============================================================
# [셀 26/26] Cloudflare Tunnel (공개 UI: https://kr-welfare-xai.com/)
# - Tunnel Public Hostname(ingress 규칙, K8s Ingress 아님): kr-welfare-xai.com → http://localhost:8000 만 연결
# - 포트 9000(RAG/XAI)은 Tunnel 대상 아님 — 셀 25 Frontend가 VM 내부 127.0.0.1:9000 호출
# - 선행: 셀 24 RAG API(9000), 셀 25 Frontend(8000) 실행 완료
# ============================================================

import os
import time

PUBLIC_UI_URL = "https://kr-welfare-xai.com/"
REQUIRED_HOSTNAME = "kr-welfare-xai.com"
REQUIRED_ORIGIN = "http://localhost:8000"

os.system("pkill -9 -f cloudflared")
time.sleep(3)
print("기존 Cloudflare tunnel 프로세스 정리 완료")

TUNNEL_TOKEN = get_setting("TUNNEL_TOKEN", "")

if not TUNNEL_TOKEN:
    raise RuntimeError("TUNNEL_TOKEN이 설정되지 않음")

os.system(
    f"nohup cloudflared tunnel --no-autoupdate run --token {TUNNEL_TOKEN} "
    "> tunnel.log 2>&1 &"
)

print("Cloudflare tunnel 연결 시도 중...")
time.sleep(8)

print("터널 실행 로그 확인:")
print("=" * 60)
log_text = ""
try:
    with open("tunnel.log", "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
        log_text = "".join(lines)
        ingress_lines = [
            ln
            for ln in lines
            if "Updated to new configuration" in ln
            or REQUIRED_HOSTNAME in ln
            or "hostname" in ln
        ]
        if ingress_lines:
            print("[Tunnel ingress / Public Hostname 관련]")
            for line in ingress_lines[-4:]:
                print(line.strip())
            print("---")
        for line in lines[-8:]:
            print(line.strip())
except Exception as e:
    print("로그 확인 중 오류:", e)
print("=" * 60)

# tunnel.log: Cloudflare Tunnel Public Hostname(ingress)에 kr-welfare-xai.com → :8000 있는지
if f'hostname\\":\\"{REQUIRED_HOSTNAME}' in log_text or f'hostname":"{REQUIRED_HOSTNAME}' in log_text:
    print(f"[OK] Tunnel ingress에 {REQUIRED_HOSTNAME} 라우트가 보입니다.")
elif REQUIRED_HOSTNAME in log_text:
    print(f"[WARN] 로그에 {REQUIRED_HOSTNAME} 문자열은 있으나 ingress 형식(Cloudflare Tunnel Public Hostname 규칙)을 확인하세요.")
else:
    print(f"[FAIL] tunnel.log에 {REQUIRED_HOSTNAME} → {REQUIRED_ORIGIN} 라우트가 없습니다.")
    print("   Cloudflare Zero Trust → Networks → Tunnels → (해당 터널) → Public Hostname:")
    print(f"     Hostname: {REQUIRED_HOSTNAME}")
    print(f"     Service:  {REQUIRED_ORIGIN}")
    print("   저장·토큰 반영 후 이 셀을 다시 실행하세요.")

if "Registered tunnel connection" in log_text:
    print("[OK] cloudflared 연결 등록됨")
else:
    print("[WARN] tunnel 연결 등록 로그가 없습니다. tunnel.log 전체를 확인하세요.")

import socket

def _port_listening(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=2):
            return True
    except OSError:
        return False

if _port_listening(8000):
    print("[OK] localhost:8000 응답 중 — Tunnel origin(Frontend) 준비됨")
else:
    print("[FAIL] localhost:8000 응답 없음 → 브라우저 502 Bad gateway")
    print("   셀 25 Frontend(8000)을 셀 24 RAG(9000) 다음에 다시 실행하세요.")
    print("   cloudflared Connected 여도 8000 미기동이면 502입니다.")

if _port_listening(9000):
    print("[OK] localhost:9000 응답 중 — RAG/XAI API (웹 /chat·내부 /ask에 필요)")
else:
    print("[FAIL] localhost:9000 응답 없음 → UI는 떠도 채팅·RAG 실패 가능")
    print("   셀 24 RAG API(9000)을 실행하세요.")

print(f"\n공개 UI 접속: {PUBLIC_UI_URL}")
print("[셀 26/26] Tunnel 설정 완료 (Tunnel=8000만 공개, RAG는 9000 내부)")
print("순서: 셀 24 RAG(9000) → 셀 25 Frontend(8000) → 셀 26 Tunnel → 브라우저")


기존 Cloudflare tunnel 프로세스 정리 완료
Cloudflare tunnel 연결 시도 중...
터널 실행 로그 확인:
[Tunnel ingress / Public Hostname 관련]
2026-09-30T00:37:14Z INF Updated to new configuration config="{\"ingress\":[{\"hostname\":\"api.kr-welfare-xai.com\",\"originRequest\":{},\"service\":\"http://localhost:9000\"},{\"hostname\":\"kr-welfare-xai.com\",\"originRequest\":{},\"service\":\"http://localhost:8000\"},{\"service\":\"http_status:404\"}],\"warp-routing\":{\"enabled\":false}}" version=4
---
2026-09-30T00:37:20Z INF precheck component="DNS Resolution" details="DNS Resolved successfully" run_id=eb199dc1-c03e-48c3-a59f-ba498e140490 status=pass target=region1.v2.argotunnel.com
2026-09-30T00:37:20Z INF precheck component="DNS Resolution" details="DNS Resolved successfully" run_id=eb199dc1-c03e-48c3-a59f-ba498e140490 status=pass target=region2.v2.argotunnel.com
2026-09-30T00:37:20Z INF precheck component="UDP Connectivity" details="QUIC connection successful" run_id=eb199dc1-c03e-48c3-a59f-ba498e140490 status=pas

In [26]:
# import os
# import subprocess
# from pathlib import Path

# print("env VLLM_ATTENTION_BACKEND:", os.environ.get("VLLM_ATTENTION_BACKEND", "(미설정, vLLM 자동 선택)"))

# log = Path("/tmp/vllm_serve.log")
# if log.is_file():
#     print("----- /tmp/vllm_serve.log -----")
#     for line in log.read_text(errors="replace").splitlines():
#         low = line.lower()
#         if any(k in low for k in ("attention", "flash", "flashinfer", "triton", "sdpa", "xformers", "backend")):
#             print(line)
# else:
#     print("로그 파일 없음. 아래 venv 조회를 봅니다.")

# py = Path("/content/vllm_env/bin/python")
# if py.is_file():
#     subprocess.run([
#         str(py), "-c",
#         "import os; print('venv env:', os.environ.get('VLLM_ATTENTION_BACKEND'));\n"
#         "import vllm, inspect, os as _os;\n"
#         "print('vllm', getattr(vllm, '__version__', '?'));\n"
#         "try:\n"
#         "    from vllm.logger import init_logger\n"
#         "except Exception:\n"
#         "    pass\n"
#         "try:\n"
#         "    from vllm.attention.selector import _cached_get_attn_backend, get_attn_backend\n"
#         "    print('get_attn_backend', get_attn_backend)\n"
#         "except Exception as e:\n"
#         "    print('selector', type(e).__name__, e)\n"
#         "try:\n"
#         "    from vllm.platforms import current_platform\n"
#         "    print('device', current_platform.get_device_name() if hasattr(current_platform, 'get_device_name') else current_platform)\n"
#         "except Exception as e:\n"
#         "    print('platform', type(e).__name__, e)\n"
#     ], check=False)